# Music Annotation Project 59R2
## Component-Invariant REMOVE Challenger

**Runtime:** CPU, High-RAM recommended. Select **Runtime → Run all**.

59R2 is a new development challenger. It does not patch, mutate, rename, or overwrite frozen 59R1. It removes `component_id` completely from the predictor schema, retains component identity only as an audit/join key, adds invariant component geometry and component-level summaries, performs page/source-group-separated out-of-fold development, cross-fits isotonic calibration, freezes a new bundle, compares 59R2 with frozen 59R1 on the complete reviewed cohort, and runs a permanent component-renumbering invariance test.

### Scientific boundary

- `component_id` is prohibited as a predictor.
- `page`, `source_group`, row order, raw coordinates, and component traversal rank are prohibited predictors.
- Clean/truth/outcome fields are labels or evaluation evidence only.
- Existing 59R1 artifacts and 60R2 outputs are read-only comparators.
- The authoritative reviewed mask remains the intervention boundary.
- Promotion authority is **false**. This notebook creates a challenger and evidence package, not a production release.

### 59R2 ordered predictor families

**Existing pixel evidence retained:** observed, normalised and paper values; mask depth; boundary distance; distance to protected evidence; edge indicator; local edge/dark/preserve/remove/ambiguous/horizontal/vertical/compact fractions; calibrated `p_print`.

**Invariant component evidence added:** component area; bounding-box width and height; aspect ratio; fill ratio; page-border contact; component maximum/mean/median depth; component preserve/remove/ambiguous/horizontal/vertical/compact fractions; component mean edge/dark fractions; component mean distance to protected evidence.

`component_id` is written to development and scoring records only as `component_key`, never placed in `X`.

In [22]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy pyarrow scikit-learn joblib opencv-python-headless matplotlib

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration, paths, and fail-closed helpers

In [23]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, random, sys, platform, math, zipfile, warnings

import cv2, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')

# Set exact run roots if automatic discovery is inappropriate.
SOURCE_59R1_DEVELOPMENT_RUN_OVERRIDE=None
SOURCE_59R1_FREEZE_RUN_OVERRIDE=None
SOURCE_60R2_RUN_OVERRIDE=None
OBSERVED_PRINT_59P_RUN_OVERRIDE=None

# Candidate roots. The notebook stops rather than guessing if no complete input exists.
DEVELOPMENT_ROOTS=[
    ROOT/'observable_residual_mask_m5_v8_7_development',
    ROOT/'remove_model_development_v8_8',
    ROOT/'remove_classifier_development',
]
FREEZE_59R1_ROOTS=[
    ROOT/'observable_residual_mask_m5_v8_8_deployment_freeze',
    ROOT/'remove_deployment_freeze_v8_8',
    ROOT/'remove_deployment_fit_and_freeze_v8_8',
]
RUN_60R2_ROOTS=[ROOT/'notebook60r2_verified_integrated_runtime',ROOT/'notebook60r2_integrated_runtime']
PRINT_59P_ROOTS=[ROOT/'observed_print_deployment_freeze_v8_2',ROOT/'observed_print_deployment_fit_and_freeze_v8_2']

OUTPUT_ROOT=ROOT/'remove_challenger_59r2_component_invariant'
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_59R2_component_invariant_challenger'
RUN=OUTPUT_ROOT/RUN_ID
INPUTS=RUN/'inputs'; TABLES=RUN/'tables'; MODELS=RUN/'models'; CALIBRATORS=RUN/'calibrators'
BUNDLE=RUN/'bundle'; REPLAY=RUN/'replay'; AUDIT=RUN/'audit'; COMPARISON=RUN/'comparison_59R1'; UPLOAD=RUN/'upload'; CHARTS=RUN/'charts'
for p in [RUN,INPUTS,TABLES,MODELS,CALIBRATORS,BUNDLE,REPLAY,AUDIT,COMPARISON,UPLOAD,CHARTS]: p.mkdir(parents=True,exist_ok=False)

SEED=42
N_SPLITS=10
REMOVE_THRESHOLD=0.995
REMOVE_CONNECTIVITY=8
REMOVE_MINIMUM_RUN=2
REMOVE_REGION_MEAN_THRESHOLD=0.995
RANDOM_RENUMBERING_SEEDS=[11,23,37,53,71,89,107,131,151,173]
EXPECTED_REVIEWED_PAGES=151
REQUIRE_EXACT_REVIEWED_COHORT=True

random.seed(SEED); np.random.seed(SEED)


def require(x,message):
    if not bool(x): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True,default=str))

def latest_complete(roots,override,required):
    if override is not None:
        p=Path(override); p=p.parent if p.name in {'bundle','artefacts'} else p
        require(all((p/r).is_file() for r in required),f'Incomplete override: {p}')
        return p
    hits=[]
    for root in roots:
        if root.exists():
            for p in root.rglob('*'):
                if p.is_dir() and all((p/r).is_file() for r in required): hits.append(p)
    require(hits,f'No complete run found below {[str(x) for x in roots]}')
    return max(hits,key=lambda p:p.stat().st_mtime)

print('59R2 output:',RUN)

59R2 output: /content/drive/MyDrive/MusicAnnotationProject/remove_challenger_59r2_component_invariant/20261112_132929_59R2_component_invariant_challenger


## 2. Predictor firewall and frozen schema

In [24]:
IDENTIFIERS=['source_group','page','component_key','component_id','y','x','row_id']
PROHIBITED_EXACT=set(IDENTIFIERS+['component','component_rank','component_order','label_id'])
PROHIBITED_PATTERNS=['clean','truth','target','outcome','failure','precision','recall','mae','safe_remove','p_remove','raw_remove_probability','structural_safe']

PIXEL_PREDICTORS=[
 'observed_value','normalised_value','paper_value','mask_depth','boundary_distance',
 'distance_to_protected','edge','local_edge_fraction','local_dark_fraction',
 'local_preserve_fraction','local_remove_fraction','local_ambiguous_fraction',
 'local_horizontal_fraction','local_vertical_fraction','local_compact_fraction','p_print'
]
COMPONENT_PREDICTORS=[
 'component_area','component_width','component_height','component_aspect_ratio','component_fill_ratio',
 'component_touches_page_border','component_maximum_mask_depth','component_mean_mask_depth','component_median_mask_depth',
 'component_preserve_fraction','component_remove_fraction','component_ambiguous_fraction',
 'component_horizontal_fraction','component_vertical_fraction','component_compact_fraction',
 'component_mean_edge_fraction','component_mean_dark_fraction','component_mean_distance_to_protected'
]
PREDICTORS=PIXEL_PREDICTORS+COMPONENT_PREDICTORS
require('component_id' not in PREDICTORS and 'component_key' not in PREDICTORS,'Identifier leakage in 59R2 schema.')
require(len(PREDICTORS)==len(set(PREDICTORS)),'Duplicate 59R2 predictors.')
for c in PREDICTORS:
    require(c not in PROHIBITED_EXACT and not any(p in c.lower() for p in PROHIBITED_PATTERNS),f'Forbidden predictor: {c}')

schema={
 'schema_version':'59R2-component-invariant-v1',
 'ordered_predictors':PREDICTORS,
 'identifier_columns':['source_group','page','component_key','y','x'],
 'prohibited_predictors':sorted(PROHIBITED_EXACT),
 'component_identity_role':'audit_join_key_only',
 'component_id_is_predictor':False,
 'predictor_count':len(PREDICTORS),
}
atomic_json(INPUTS/'predictor_schema.json',schema)
print('59R2 predictors:',len(PREDICTORS))

59R2 predictors: 34


## 3. Resolve development rows, frozen 59R1, and 60R2 cohort

In [25]:
# ============================================================
# Cell 3 - Resolve exact 59R1 training lineage and 60R2 source
# ============================================================

# 59R2 must challenge the exact development population used by
# frozen 59R1. Do not search for guessed development filenames.
#
# Authoritative lineage:
#
#   59R1
#     audit/training_population_identity.json
#                |
#                +--> Notebook 49:
#                |    03_truth_unlock/pixel_oracle_gap.parquet
#                |
#                +--> Notebook 50:
#                     tables/all_m3_oof_predictions_and_decisions.parquet
#
# Both source-table hashes are recorded by 59R1 and are verified
# below before any development data are loaded.


# ------------------------------------------------------------
# 1. Resolve exact completed incumbent 59R1 freeze
# ------------------------------------------------------------

R59 = latest_complete(
    FREEZE_59R1_ROOTS,
    SOURCE_59R1_FREEZE_RUN_OVERRIDE,
    [
        'audit/training_population_identity.json',
        'bundle/remove_model.joblib',
        'bundle/remove_calibrator.joblib',
        'bundle/predictor_schema.json',
        'bundle/policy.json',
        'bundle/freeze_manifest.json',
        'replay/replay_test.json',
    ],
)

require(
    R59 is not None,
    (
        'No complete Notebook 59R1 freeze run was found. '
        'Set SOURCE_59R1_FREEZE_RUN_OVERRIDE to the exact '
        'completed 59R1 run if necessary.'
    ),
)


# ------------------------------------------------------------
# 2. Read the exact 59R1 development-population identity
# ------------------------------------------------------------

POPULATION_IDENTITY_PATH = (
    R59 / 'audit' / 'training_population_identity.json'
)

require(
    POPULATION_IDENTITY_PATH.is_file(),
    (
        '59R1 training_population_identity.json is missing: '
        f'{POPULATION_IDENTITY_PATH}'
    ),
)

population_identity = json.loads(
    POPULATION_IDENTITY_PATH.read_text()
)

required_population_fields = {
    'source_notebook49_run',
    'source_notebook50_run',
    'source_pixel_table',
    'source_pixel_table_sha256',
    'notebook50_oof_table',
    'notebook50_oof_table_sha256',
    'rows',
    'pages',
    'source_groups',
    'ordered_predictors',
    'identity_columns',
    'label_column',
    'raw_remove_oof_column',
}

missing_population_fields = (
    required_population_fields
    - set(population_identity)
)

require(
    not missing_population_fields,
    (
        '59R1 training population identity is incomplete. '
        f'Missing fields: {sorted(missing_population_fields)}'
    ),
)


# ------------------------------------------------------------
# 3. Resolve exact Notebook 49 / Notebook 50 lineage
# ------------------------------------------------------------

NB49 = Path(
    population_identity['source_notebook49_run']
)

NB50 = Path(
    population_identity['source_notebook50_run']
)

DEV_PATH = Path(
    population_identity['source_pixel_table']
)

NB50_OOF_PATH = Path(
    population_identity['notebook50_oof_table']
)

# Preserve DEV_RUN for downstream cells that still refer to it.
DEV_RUN = NB49


# ------------------------------------------------------------
# 4. Fail closed if historical source paths have disappeared
# ------------------------------------------------------------

require(
    NB49.is_dir(),
    f'59R1 source Notebook 49 run no longer exists: {NB49}',
)

require(
    NB50.is_dir(),
    f'59R1 source Notebook 50 run no longer exists: {NB50}',
)

require(
    DEV_PATH.is_file(),
    (
        'Exact 59R1 Notebook 49 development pixel table '
        f'no longer exists: {DEV_PATH}'
    ),
)

require(
    NB50_OOF_PATH.is_file(),
    (
        'Exact 59R1 Notebook 50 OOF table '
        f'no longer exists: {NB50_OOF_PATH}'
    ),
)


# ------------------------------------------------------------
# 5. Verify exact historical table hashes
# ------------------------------------------------------------

expected_dev_sha256 = str(
    population_identity['source_pixel_table_sha256']
)

actual_dev_sha256 = sha256_file(
    DEV_PATH
)

require(
    actual_dev_sha256 == expected_dev_sha256,
    (
        'Notebook 49 development population hash mismatch.\n'
        f'Path: {DEV_PATH}\n'
        f'Expected: {expected_dev_sha256}\n'
        f'Actual:   {actual_dev_sha256}\n'
        '59R2 refuses to train against a population different '
        'from the one used by 59R1.'
    ),
)

expected_oof_sha256 = str(
    population_identity['notebook50_oof_table_sha256']
)

actual_oof_sha256 = sha256_file(
    NB50_OOF_PATH
)

require(
    actual_oof_sha256 == expected_oof_sha256,
    (
        'Notebook 50 OOF evidence hash mismatch.\n'
        f'Path: {NB50_OOF_PATH}\n'
        f'Expected: {expected_oof_sha256}\n'
        f'Actual:   {actual_oof_sha256}\n'
        '59R2 refuses to continue against changed incumbent '
        'development evidence.'
    ),
)


# ------------------------------------------------------------
# 6. Verify the historical files have the expected identities
# ------------------------------------------------------------

development_probe = pd.read_parquet(
    DEV_PATH
)

oof_probe = pd.read_parquet(
    NB50_OOF_PATH
)

identity_columns = [
    str(x)
    for x in population_identity['identity_columns']
]

require(
    set(identity_columns).issubset(development_probe.columns),
    (
        'Notebook 49 development table lacks recorded identity '
        f'columns: {sorted(set(identity_columns) - set(development_probe.columns))}'
    ),
)

require(
    set(identity_columns).issubset(oof_probe.columns),
    (
        'Notebook 50 OOF table lacks recorded identity '
        f'columns: {sorted(set(identity_columns) - set(oof_probe.columns))}'
    ),
)

require(
    not development_probe.duplicated(identity_columns).any(),
    'Notebook 49 development population has duplicate identity rows.',
)

require(
    not oof_probe.duplicated(identity_columns).any(),
    'Notebook 50 OOF evidence has duplicate identity rows.',
)

require(
    len(development_probe)
    == int(population_identity['rows']),
    (
        'Notebook 49 row count no longer matches the '
        '59R1 training population identity.'
    ),
)

require(
    development_probe['page'].astype(str).nunique()
    == int(population_identity['pages']),
    (
        'Notebook 49 page count no longer matches the '
        '59R1 training population identity.'
    ),
)

require(
    development_probe['source_group'].astype(str).nunique()
    == int(population_identity['source_groups']),
    (
        'Notebook 49 source-group count no longer matches the '
        '59R1 training population identity.'
    ),
)


# ------------------------------------------------------------
# 7. Explicitly prove what 59R2 is changing
# ------------------------------------------------------------

incumbent_predictors = [
    str(x)
    for x in population_identity['ordered_predictors']
]

require(
    'component_id' in incumbent_predictors,
    (
        'The resolved 59R1 training identity does not contain '
        'component_id. That would contradict the defect that '
        '59R2 is intended to challenge, so stop rather than '
        'silently running a different experiment.'
    ),
)

require(
    population_identity['label_column']
    in development_probe.columns,
    (
        'Recorded 59R1 truth-side label column is absent from '
        'the Notebook 49 development population.'
    ),
)

require(
    population_identity['raw_remove_oof_column']
    in oof_probe.columns,
    (
        'Recorded 59R1 raw REMOVE OOF column is absent from '
        'the Notebook 50 evidence table.'
    ),
)


# Free the temporary probes. Downstream cells can load the
# authoritative tables when required.
del development_probe
del oof_probe


# ------------------------------------------------------------
# 8. Resolve completed 60R2 operational run
# ------------------------------------------------------------

R60 = latest_complete(
    RUN_60R2_ROOTS,
    SOURCE_60R2_RUN_OVERRIDE,
    [
        'verdict.json',
        'tables/pixel_predictions.parquet',
        'tables/page_summary.csv',
    ],
)

require(
    R60 is not None,
    (
        'No complete 60R2 run was found. '
        'Set SOURCE_60R2_RUN_OVERRIDE if necessary.'
    ),
)


# ------------------------------------------------------------
# 9. Record every immutable source used by 59R2
# ------------------------------------------------------------

source_files = {
    # Authoritative 59R1 lineage
    '59r1_training_population_identity':
        POPULATION_IDENTITY_PATH,

    'notebook49_development_pixel_table':
        DEV_PATH,

    'notebook50_oof_evidence':
        NB50_OOF_PATH,

    # Frozen incumbent 59R1
    '59r1_model':
        R59 / 'bundle' / 'remove_model.joblib',

    '59r1_calibrator':
        R59 / 'bundle' / 'remove_calibrator.joblib',

    '59r1_schema':
        R59 / 'bundle' / 'predictor_schema.json',

    '59r1_policy':
        R59 / 'bundle' / 'policy.json',

    '59r1_freeze_manifest':
        R59 / 'bundle' / 'freeze_manifest.json',

    '59r1_replay':
        R59 / 'replay' / 'replay_test.json',

    # Completed 60R2 operational comparison
    '60r2_pixels':
        R60 / 'tables' / 'pixel_predictions.parquet',

    '60r2_page_summary':
        R60 / 'tables' / 'page_summary.csv',

    '60r2_verdict':
        R60 / 'verdict.json',
}

for name, path in source_files.items():
    require(
        Path(path).is_file(),
        f'Missing required 59R2 source artefact {name}: {path}',
    )


# ------------------------------------------------------------
# 10. Freeze pre-run source hashes
# ------------------------------------------------------------

atomic_json(
    AUDIT / 'source_hashes_before.json',
    {
        name: {
            'path': str(path),
            'sha256': sha256_file(path),
        }
        for name, path in source_files.items()
    },
)


# ------------------------------------------------------------
# 11. Write explicit lineage record for 59R2
# ------------------------------------------------------------

lineage_record = {
    'source_59r1_run':
        str(R59),

    'source_notebook49_run':
        str(NB49),

    'source_notebook50_run':
        str(NB50),

    'source_notebook49_pixel_table':
        str(DEV_PATH),

    'source_notebook49_pixel_table_sha256':
        actual_dev_sha256,

    'source_notebook50_oof_table':
        str(NB50_OOF_PATH),

    'source_notebook50_oof_table_sha256':
        actual_oof_sha256,

    'source_60r2_run':
        str(R60),

    '59r1_training_rows':
        int(population_identity['rows']),

    '59r1_training_pages':
        int(population_identity['pages']),

    '59r1_training_source_groups':
        int(population_identity['source_groups']),

    '59r1_predictor_count':
        len(incumbent_predictors),

    '59r1_ordered_predictors':
        incumbent_predictors,

    '59r1_label_column':
        str(population_identity['label_column']),

    '59r1_raw_remove_oof_column':
        str(population_identity['raw_remove_oof_column']),

    '59r2_schema_change':
        {
            'removed_predictor': 'component_id',
            'component_identity_role':
                'audit/join key only; prohibited predictor',
        },

    'historical_population_changed':
        False,

    'historical_table_hashes_verified':
        True,
}

atomic_json(
    AUDIT / 'source_lineage.json',
    lineage_record,
)


# ------------------------------------------------------------
# 12. Console summary
# ------------------------------------------------------------

print('59R1 incumbent:', R59)
print('Notebook 49 development source:', NB49)
print('Notebook 50 OOF source:', NB50)
print('Development table:', DEV_PATH)
print('Notebook 50 OOF table:', NB50_OOF_PATH)
print('60R2:', R60)

print(
    '59R1 development population:',
    population_identity['rows'],
    'rows /',
    population_identity['pages'],
    'pages /',
    population_identity['source_groups'],
    'source groups'
)

print(
    '59R1 predictors:',
    len(incumbent_predictors)
)

print(
    '59R2 intentional schema change: '
    'component_id removed from predictors; '
    'component identity retained only for audit/join purposes.'
)

59R1 incumbent: /content/drive/MyDrive/MusicAnnotationProject/observable_residual_mask_m5_v8_8_deployment_freeze/20261111_131009_notebook59r1_remove_deployment_freeze
Notebook 49 development source: /content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_7_notebook49/20261029_085706_v8_7_notebook49
Notebook 50 OOF source: /content/drive/MyDrive/MusicAnnotationProject/observable_residual_mask_m5_v8_7/20261101_215506_notebook50_v8_7_asymmetric_m5
Development table: /content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_7_notebook49/20261029_085706_v8_7_notebook49/03_truth_unlock/pixel_oracle_gap.parquet
Notebook 50 OOF table: /content/drive/MyDrive/MusicAnnotationProject/observable_residual_mask_m5_v8_7/20261101_215506_notebook50_v8_7_asymmetric_m5/tables/all_m3_oof_predictions_and_decisions.parquet
60R2: /content/drive/MyDrive/MusicAnnotationProject/notebook60r2_verified_integrated_runtime/20261112_093037_notebook60r2
59R1 development population: 3

## 4. Load and normalise the labelled development dataset

In [27]:
# ============================================================
# Cell 4 - Load exact 59R1 development population and
#          reconstruct the incumbent binary REMOVE target
# ============================================================

# DEV_PATH was resolved in Cell 3 from the exact historical
# population recorded by:
#
#   59R1/audit/training_population_identity.json
#
# Do NOT search for a new training table here.
# Do NOT assume the historical label column is already 0/1.

dev = pd.read_parquet(
    DEV_PATH
).copy()

require(
    len(dev) > 0,
    '59R1 development population is empty.',
)


# ------------------------------------------------------------
# 1. Verify population identity against the frozen 59R1 record
# ------------------------------------------------------------

require(
    len(dev)
    == int(population_identity['rows']),
    (
        'Development row count differs from frozen 59R1 '
        'training_population_identity.json.'
    ),
)

require(
    'page' in dev.columns,
    'Development population lacks page.',
)

require(
    'source_group' in dev.columns,
    'Development population lacks source_group.',
)

require(
    dev['page'].astype(str).nunique()
    == int(population_identity['pages']),
    (
        'Development page count differs from frozen '
        '59R1 population identity.'
    ),
)

require(
    dev['source_group'].astype(str).nunique()
    == int(population_identity['source_groups']),
    (
        'Development source-group count differs from '
        'frozen 59R1 population identity.'
    ),
)


# ------------------------------------------------------------
# 2. Resolve the exact incumbent truth-side label column
# ------------------------------------------------------------

recorded_label_column = str(
    population_identity['label_column']
)

require(
    recorded_label_column in dev.columns,
    (
        'The truth-side label column recorded by 59R1 '
        'is absent from the Notebook 49 development table: '
        f'{recorded_label_column}'
    ),
)

LABEL = recorded_label_column


# ------------------------------------------------------------
# 3. Inspect the historical labels BEFORE deriving the target
# ------------------------------------------------------------

raw_labels = (
    dev[LABEL]
    .astype(str)
    .str.strip()
    .str.upper()
)

label_counts = (
    raw_labels
    .value_counts(
        dropna=False
    )
    .rename_axis(
        'historical_label'
    )
    .reset_index(
        name='rows'
    )
)

label_counts.to_csv(
    AUDIT / 'historical_label_counts.csv',
    index=False,
)

print(
    'Historical label column:',
    LABEL,
)

print(
    '\nHistorical label values:'
)

display(
    label_counts
)


# ------------------------------------------------------------
# 4. Reconstruct EXACTLY the binary target used by 59R1
#
# Positive class:
#
#     historical label == REMOVE
#
# Everything else is the non-REMOVE class.
#
# Critically, do NOT use:
#
#     pd.to_numeric(label)
#
# because the historical label is semantic, not a preconstructed
# binary model target.
# ------------------------------------------------------------

y = (
    raw_labels
    == 'REMOVE'
).astype(
    np.uint8
)


target_counts = (
    pd.Series(
        y,
        name='remove_target',
    )
    .value_counts()
    .sort_index()
    .rename_axis(
        'remove_target'
    )
    .reset_index(
        name='rows'
    )
)

target_counts[
    'meaning'
] = target_counts[
    'remove_target'
].map({
    0:
        'NOT_REMOVE',

    1:
        'REMOVE',
})

target_counts.to_csv(
    AUDIT / 'remove_target_counts.csv',
    index=False,
)


print(
    '\nReconstructed incumbent REMOVE target:'
)

display(
    target_counts
)


# ------------------------------------------------------------
# 5. Fail closed if the exact incumbent population really is
#    one-class after semantic target reconstruction
# ------------------------------------------------------------

require(
    y.nunique() == 2,
    (
        'The exact 59R1 development population becomes '
        'one-class after reconstructing the target as '
        f'{LABEL} == REMOVE. '
        f'Observed target values: {sorted(y.unique().tolist())}. '
        'This would contradict the incumbent 59R1 fit and must '
        'be investigated rather than bypassed.'
    ),
)


remove_rows = int(
    y.sum()
)

not_remove_rows = int(
    len(y) - remove_rows
)

require(
    remove_rows > 0,
    'No REMOVE-positive rows were reconstructed.',
)

require(
    not_remove_rows > 0,
    'No non-REMOVE rows were reconstructed.',
)


print(
    f'\nREMOVE positives: {remove_rows:,}'
)

print(
    f'NOT_REMOVE rows: {not_remove_rows:,}'
)

print(
    'REMOVE prevalence:',
    f'{y.mean():.6%}',
)


# ------------------------------------------------------------
# 6. Resolve component identity strictly as an AUDIT/JOIN KEY
#
# This is the defect being corrected in 59R2:
#
#     component identity must NOT enter X.
# ------------------------------------------------------------

if 'component_key' not in dev.columns:

    if 'component_id' in dev.columns:

        dev[
            'component_key'
        ] = (
            dev[
                'component_id'
            ]
            .astype(str)
        )

        component_identity_source = (
            'component_id'
        )

    elif 'component' in dev.columns:

        dev[
            'component_key'
        ] = (
            dev[
                'component'
            ]
            .astype(str)
        )

        component_identity_source = (
            'component'
        )

    else:

        raise RuntimeError(
            'Development population has no component identity '
            'field. Expected component_id or component for '
            'audit/join use only.'
        )

else:

    dev[
        'component_key'
    ] = (
        dev[
            'component_key'
        ]
        .astype(str)
    )

    component_identity_source = (
        'component_key'
    )


# ------------------------------------------------------------
# 7. Normalise identity fields
# ------------------------------------------------------------

dev[
    'page'
] = (
    dev[
        'page'
    ]
    .astype(str)
)

dev[
    'source_group'
] = (
    dev[
        'source_group'
    ]
    .astype(str)
)

dev[
    'component_key'
] = (
    dev[
        'component_key'
    ]
    .astype(str)
)


require(
    not dev[
        'page'
    ].str.strip().eq('').any(),
    'Development population contains blank page identity.',
)

require(
    not dev[
        'source_group'
    ].str.strip().eq('').any(),
    'Development population contains blank source_group.',
)

require(
    not dev[
        'component_key'
    ].str.strip().eq('').any(),
    'Development population contains blank component identity.',
)


# ------------------------------------------------------------
# 8. Verify coordinates required for invariant component
#    geometry construction
# ------------------------------------------------------------

require(
    'y' in dev.columns,
    (
        'Development population lacks y. '
        '59R2 requires row coordinates only to reconstruct '
        'component geometry; y will never enter the model.'
    ),
)

require(
    'x' in dev.columns,
    (
        'Development population lacks x. '
        '59R2 requires row coordinates only to reconstruct '
        'component geometry; x will never enter the model.'
    ),
)


dev[
    'y'
] = pd.to_numeric(
    dev[
        'y'
    ],
    errors='raise',
).astype(
    np.int32
)

dev[
    'x'
] = pd.to_numeric(
    dev[
        'x'
    ],
    errors='raise',
).astype(
    np.int32
)


require(
    (
        dev[
            'y'
        ]
        >= 0
    ).all(),
    'Negative y coordinate found.',
)

require(
    (
        dev[
            'x'
        ]
        >= 0
    ).all(),
    'Negative x coordinate found.',
)


# ------------------------------------------------------------
# 9. Verify that component_id remains prohibited as a predictor
# ------------------------------------------------------------

require(
    'component_id'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'component_id remains in PREDICTORS.'
    ),
)

require(
    'component_key'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'component_key remains in PREDICTORS.'
    ),
)

require(
    'component'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'component remains in PREDICTORS.'
    ),
)

require(
    'page'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'page remains in PREDICTORS.'
    ),
)

require(
    'source_group'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'source_group remains in PREDICTORS.'
    ),
)

require(
    'x'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'x remains in PREDICTORS.'
    ),
)

require(
    'y'
    not in PREDICTORS,
    (
        '59R2 predictor firewall failure: '
        'y remains in PREDICTORS.'
    ),
)


# ------------------------------------------------------------
# 10. Check currently available pixel predictors
#
# The next cell will add the new invariant component-level
# predictors. At this point we require only the pixel evidence
# that must already exist in the 59R1 population.
# ------------------------------------------------------------

missing_pixel_predictors = [
    predictor
    for predictor in PIXEL_PREDICTORS
    if predictor not in dev.columns
]

require(
    not missing_pixel_predictors,
    (
        'The exact 59R1 Notebook 49 population lacks '
        'pixel predictors required by the current 59R2 '
        'schema: '
        f'{missing_pixel_predictors}'
    ),
)


# ------------------------------------------------------------
# 11. Numeric integrity of pre-existing observable predictors
# ------------------------------------------------------------

for predictor in PIXEL_PREDICTORS:

    values = pd.to_numeric(
        dev[
            predictor
        ],
        errors='coerce',
    )

    require(
        not np.isinf(
            values.to_numpy(
                dtype=np.float64
            )
        ).any(),
        (
            'Infinite development predictor values found '
            f'in {predictor}.'
        ),
    )


# ------------------------------------------------------------
# 12. Write explicit target-construction audit
# ------------------------------------------------------------

target_record = {
    'source_59r1_run':
        str(R59),

    'source_notebook49_run':
        str(NB49),

    'development_table':
        str(DEV_PATH),

    'development_table_sha256':
        sha256_file(
            DEV_PATH
        ),

    'historical_label_column':
        LABEL,

    'historical_label_values':
        sorted(
            raw_labels
            .unique()
            .tolist()
        ),

    'remove_target_rule':
        (
            f"UPPER(TRIM({LABEL})) == 'REMOVE'"
        ),

    'remove_positive_rows':
        remove_rows,

    'not_remove_rows':
        not_remove_rows,

    'remove_prevalence':
        float(
            y.mean()
        ),

    'component_identity_source':
        component_identity_source,

    'component_identity_role':
        (
            'audit/join key only; '
            'explicitly excluded from predictor matrix'
        ),

    'component_id_predictor':
        False,

    'coordinates_predictors':
        False,

    'truth_used_as_predictor':
        False,
}

atomic_json(
    AUDIT / 'target_construction.json',
    target_record,
)


# ---------------------------

Historical label column: diagnostic_class

Historical label values:


,historical_label,rows
0,REMOVE,25469
1,UNCERTAIN,8085
2,PROTECT,890



Reconstructed incumbent REMOVE target:


,remove_target,rows,meaning
0,0,8975,NOT_REMOVE
1,1,25469,REMOVE


REMOVE positives: 25,469
NOT_REMOVE rows: 8,975
REMOVE prevalence: 73.943212%


## 5. Rebuild legitimate invariant component geometry and summaries

In [28]:
def add_component_features(frame):
    f=frame.copy()
    rows=[]
    H_col=next((c for c in ['page_height','height','image_height'] if c in f.columns),None)
    W_col=next((c for c in ['page_width','width','image_width'] if c in f.columns),None)
    for (page,ck),g in f.groupby(['page','component_key'],sort=False):
        ys=pd.to_numeric(g.y,errors='raise').to_numpy(int); xs=pd.to_numeric(g.x,errors='raise').to_numpy(int)
        area=len(g); x0,x1=int(xs.min()),int(xs.max())+1; y0,y1=int(ys.min()),int(ys.max())+1
        w=x1-x0; h=y1-y0
        page_h=int(g[H_col].iloc[0]) if H_col else int(f.loc[f.page==page,'y'].max())+1
        page_w=int(g[W_col].iloc[0]) if W_col else int(f.loc[f.page==page,'x'].max())+1
        def mean(c): return float(pd.to_numeric(g[c],errors='coerce').mean())
        rows.append({
          'page':page,'component_key':ck,
          'component_area':int(area),'component_width':int(w),'component_height':int(h),
          'component_aspect_ratio':float(max(w,h)/max(min(w,h),1)),
          'component_fill_ratio':float(area/max(w*h,1)),
          'component_touches_page_border':int(x0==0 or y0==0 or x1>=page_w or y1>=page_h),
          'component_maximum_mask_depth':float(pd.to_numeric(g.mask_depth,errors='coerce').max()),
          'component_mean_mask_depth':mean('mask_depth'),'component_median_mask_depth':float(pd.to_numeric(g.mask_depth,errors='coerce').median()),
          'component_preserve_fraction':mean('local_preserve_fraction'),'component_remove_fraction':mean('local_remove_fraction'),
          'component_ambiguous_fraction':mean('local_ambiguous_fraction'),'component_horizontal_fraction':mean('local_horizontal_fraction'),
          'component_vertical_fraction':mean('local_vertical_fraction'),'component_compact_fraction':mean('local_compact_fraction'),
          'component_mean_edge_fraction':mean('local_edge_fraction'),'component_mean_dark_fraction':mean('local_dark_fraction'),
          'component_mean_distance_to_protected':mean('distance_to_protected'),
        })
    comp=pd.DataFrame(rows)
    require(not comp.duplicated(['page','component_key']).any(),'Duplicate component feature records.')
    return f.drop(columns=[c for c in COMPONENT_PREDICTORS if c in f.columns],errors='ignore').merge(comp,on=['page','component_key'],validate='many_to_one'),comp

dev,development_components=add_component_features(dev)
development_components.to_parquet(TABLES/'development_component_features.parquet',index=False)
missing=[c for c in PREDICTORS if c not in dev.columns]; require(not missing,f'59R2 predictors missing after rebuild: {missing}')
X=dev[PREDICTORS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
require(not np.isinf(X.to_numpy(float)).any(),'Infinite development predictor value.')

# Predictor firewall: exact schema and no hidden identifiers.
require(list(X.columns)==PREDICTORS,'Ordered schema mismatch.')
for forbidden in PROHIBITED_EXACT:
    require(forbidden not in X.columns,f'Forbidden predictor entered X: {forbidden}')
shutil.copy2(DEV_PATH,INPUTS/DEV_PATH.name)
dev[['source_group','page','component_key','y','x',LABEL]+PREDICTORS].to_parquet(TABLES/'59r2_development_dataset.parquet',index=False)
print('Built invariant component predictors:',len(development_components))

Built invariant component predictors: 3719


## 6. Page/source-group-separated fold map

In [29]:
# Prefer source-group separation. Page identity is additionally asserted disjoint.
groups=dev.source_group.astype(str)
unique_groups=groups.nunique()
require(unique_groups>=3,'At least three source groups are required.')
splitter=LeaveOneGroupOut() if unique_groups<=10 else GroupKFold(n_splits=min(N_SPLITS,unique_groups))
fold=np.full(len(dev),-1,int); split_rows=[]
for f,(tr,te) in enumerate(splitter.split(X,y,groups),1):
    train_groups=set(groups.iloc[tr]); test_groups=set(groups.iloc[te]); train_pages=set(dev.page.iloc[tr]); test_pages=set(dev.page.iloc[te])
    require(not train_groups&test_groups,f'Fold {f}: source-group leakage.')
    require(not train_pages&test_pages,f'Fold {f}: page leakage.')
    fold[te]=f
    split_rows.append({'fold':f,'train_rows':len(tr),'test_rows':len(te),'train_pages':len(train_pages),'test_pages':len(test_pages),'train_groups':len(train_groups),'test_groups':len(test_groups),'test_group_names':'|'.join(sorted(test_groups))})
require((fold>0).all(),'Incomplete fold assignment.')
dev['fold']=fold
pd.DataFrame(split_rows).to_csv(INPUTS/'fold_map.csv',index=False)
print(pd.DataFrame(split_rows).to_string(index=False))

 fold  train_rows  test_rows  train_pages  test_pages  train_groups  test_groups                                         test_group_names
    1       27541       6903           38           5            16            1                                   Anglo_Music_Press_HotH
    2       30283       4161           40           3            16            1                               Wright_and_Round_Somewhere
    3       30878       3566           40           3            16            1                                            Cantino_WWTLF
    4       31157       3287           39           4            16            1                                          Whelan_Assassin
    5       31513       2931           41           2            16            1                                           Cantino_Facade
    6       31786       2658           41           2            16            1                                         Bernaerts_Maniac
    7       31783       2661      

## 7. Candidate models and source-group OOF development

In [ ]:
MODELS={
 'logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]),
 'forest':Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=600,min_samples_leaf=10,max_features='sqrt',class_weight='balanced_subsample',random_state=SEED,n_jobs=-1))]),
 'hist_gradient':Pipeline([('impute',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=2.0,random_state=SEED))]),
}

summaries=[]; fold_results=[]; raw_by_model={}
for name,template in MODELS.items():
    raw=np.full(len(dev),np.nan)
    for f in sorted(np.unique(fold)):
        tr=np.where(fold!=f)[0]; te=np.where(fold==f)[0]
        require(y.iloc[tr].nunique()==2,f'{name} fold {f}: one-class training set.')
        m=clone(template).fit(X.iloc[tr],y.iloc[tr]); raw[te]=m.predict_proba(X.iloc[te])[:,1]
        yt=y.iloc[te].to_numpy(); pt=raw[te]
        fold_results.append({'model':name,'fold':int(f),'test_rows':len(te),'test_pages':dev.page.iloc[te].nunique(),'test_groups':dev.source_group.iloc[te].nunique(),'roc_auc':roc_auc_score(yt,pt) if len(np.unique(yt))==2 else np.nan,'average_precision':average_precision_score(yt,pt),'brier':brier_score_loss(yt,pt)})
    require(np.isfinite(raw).all(),f'{name}: incomplete OOF predictions.')
    raw_by_model[name]=raw
    summaries.append({'model':name,'roc_auc':roc_auc_score(y,raw),'average_precision':average_precision_score(y,raw),'brier':brier_score_loss(y,raw)})
summary_df=pd.DataFrame(summaries).sort_values(['average_precision','roc_auc'],ascending=False)
summary_df.to_csv(TABLES/'model_comparison.csv',index=False); pd.DataFrame(fold_results).to_csv(TABLES/'fold_results.csv',index=False)
BEST=str(summary_df.iloc[0].model)
print('Selected model family:',BEST); display(summary_df)

## 8. Cross-fitted isotonic calibration and frozen decision policy

In [ ]:
raw=raw_by_model[BEST]
calibrated=np.full(len(dev),np.nan)
for f in sorted(np.unique(fold)):
    tr=np.where(fold!=f)[0]; te=np.where(fold==f)[0]
    iso=IsotonicRegression(out_of_bounds='clip').fit(raw[tr],y.iloc[tr].to_numpy())
    calibrated[te]=iso.predict(raw[te])
require(np.isfinite(calibrated).all(),'Incomplete cross-fitted calibration.')

dev['raw_59r2_oof']=raw; dev['p_remove_59r2_oof']=calibrated; dev['provisional_remove_59r2']=calibrated>=REMOVE_THRESHOLD
dev[['source_group','page','component_key','y','x','fold',LABEL,'raw_59r2_oof','p_remove_59r2_oof','provisional_remove_59r2']].to_parquet(TABLES/'59r2_oof_predictions.parquet',index=False)

metrics={'model':BEST,'roc_auc_raw':float(roc_auc_score(y,raw)),'average_precision_raw':float(average_precision_score(y,raw)),'brier_raw':float(brier_score_loss(y,raw)),'brier_calibrated':float(brier_score_loss(y,calibrated)),'threshold':REMOVE_THRESHOLD,'selected_pixels':int((calibrated>=REMOVE_THRESHOLD).sum()),'selected_positive':int(((calibrated>=REMOVE_THRESHOLD)&(y.to_numpy()==1)).sum()),'selected_negative':int(((calibrated>=REMOVE_THRESHOLD)&(y.to_numpy()==0)).sum())}
atomic_json(RUN/'development_metrics.json',metrics)
print(json.dumps(metrics,indent=2))

## 9. Refit chosen 59R2 estimator and calibrator on permitted development evidence

In [ ]:
final_model=clone(MODELS[BEST]).fit(X,y)
# Calibration is fitted to OOF raw scores, never in-sample fitted scores.
final_calibrator=IsotonicRegression(out_of_bounds='clip').fit(raw,y.to_numpy())
joblib.dump(final_model,BUNDLE/'remove_model.joblib')
joblib.dump(final_calibrator,BUNDLE/'remove_calibrator.joblib')
shutil.copy2(INPUTS/'predictor_schema.json',BUNDLE/'predictor_schema.json')

policy_payload={
 'policy_version':'59R2-component-invariant-v1','policy_id':None,'development_only':True,'promotion_authority':False,
 'threshold':REMOVE_THRESHOLD,'remove_threshold':REMOVE_THRESHOLD,'connectivity':REMOVE_CONNECTIVITY,
 'minimum_run':REMOVE_MINIMUM_RUN,'region_mean_threshold':REMOVE_REGION_MEAN_THRESHOLD,
 'safe_protect_enabled':False,'component_id_is_predictor':False,'component_identity_role':'audit_join_key_only',
 'model_family':BEST,'calibration':'isotonic fitted to source-group OOF raw scores','ordered_predictors':PREDICTORS,
}
policy_payload['policy_id']='59R2-'+object_hash(policy_payload)[:16]
atomic_json(BUNDLE/'policy.json',policy_payload)
print('Policy ID:',policy_payload['policy_id'])

## 10. Exact replay pack

In [ ]:
# Deterministic multi-page replay sample.
rng=np.random.default_rng(SEED)
replay_idx=[]
for page,g in dev.groupby('page'):
    idx=g.index.to_numpy(); replay_idx.extend(rng.choice(idx,min(25,len(idx)),replace=False).tolist())
replay_idx=np.array(sorted(replay_idx),int)
replay_X=X.loc[replay_idx].reset_index(drop=True)
replay_meta=dev.loc[replay_idx,['source_group','page','component_key','y','x']].reset_index(drop=True)
replay_raw=np.asarray(final_model.predict_proba(replay_X)[:,1],np.float64)
replay_prob=np.asarray(final_calibrator.predict(replay_raw),np.float64)
replay_decision=replay_prob>=REMOVE_THRESHOLD
replay_meta.to_csv(REPLAY/'replay_rows.csv',index=False); replay_X.to_parquet(REPLAY/'replay_predictors.parquet',index=False)
np.savez_compressed(REPLAY/'replay_expected.npz',raw=replay_raw,probability=replay_prob,decision=replay_decision.astype(np.uint8))

m2=joblib.load(BUNDLE/'remove_model.joblib'); c2=joblib.load(BUNDLE/'remove_calibrator.joblib')
r2=np.asarray(m2.predict_proba(replay_X)[:,1],np.float64); p2=np.asarray(c2.predict(r2),np.float64); d2=p2>=REMOVE_THRESHOLD
replay_result={'passed':bool(np.array_equal(r2,replay_raw) and np.array_equal(p2,replay_prob) and np.array_equal(d2,replay_decision)),'model_reload_exact':bool(np.array_equal(r2,replay_raw)),'calibrator_reload_exact':bool(np.array_equal(p2,replay_prob)),'decision_reload_exact':bool(np.array_equal(d2,replay_decision)),'rows':len(replay_X),'policy_id':policy_payload['policy_id']}
atomic_json(REPLAY/'replay_test.json',replay_result); require(replay_result['passed'],'59R2 replay failed.')
print(json.dumps(replay_result,indent=2))

## 11. Score complete 60R2 reviewed cohort with 59R2 and compare frozen 59R1

In [ ]:
# ============================================================
# Cell 11 - Memory-bounded complete-cohort scoring
#
# Corrected implementation for the actual 60R2 parquet schema.
#
# Key properties:
#   1. Loads the complete 60R2 cohort exactly once.
#   2. Uses component_id only to identify component membership.
#   3. Computes invariant component features once per
#      (page, component_key).
#   4. Does NOT join component features back onto the 12.7M-row
#      DataFrame.
#   5. Maps component features directly into bounded float32
#      NumPy inference batches.
#   6. Does NOT construct another full Pandas predictor frame.
#   7. Uses the already persisted frozen 59R1 predictions rather
#      than unnecessarily rescoring 59R1.
#   8. Scores 59R2 in bounded batches and writes results to
#      local float32 memmaps.
#
# Scientific contract:
#   - no sampling;
#   - no predictor removal;
#   - no threshold movement;
#   - no model or calibrator refit;
#   - no use of component_id as a predictor;
#   - no change to frozen 59R1 output.
# ============================================================

import gc
import json
import time
import resource
from pathlib import Path

import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

LOCAL_WORK = Path('/content/59r2_cell11_work')
LOCAL_WORK.mkdir(
    parents=True,
    exist_ok=True,
)

# Reduce to 100,000 if Colab still approaches its RAM limit.
SCORE_BATCH_ROWS = 250_000

COMPONENT_KEYS = [
    'page',
    'component_key',
]

PROHIBITED_59R2_PREDICTORS = {
    'component_id',
    'component',
    'component_key',
    'component_rank',
    'component_order',
    'page',
    'source_group',
    'x',
    'y',
    'row_id',
}


# ------------------------------------------------------------
# 1. Resource helpers
# ------------------------------------------------------------

def resident_gib():
    """
    Peak resident memory in GiB on Linux/Colab.
    """
    return (
        resource.getrusage(
            resource.RUSAGE_SELF
        ).ru_maxrss
        / 1024.0
        / 1024.0
    )


def frame_gib(frame):
    return (
        frame.memory_usage(
            index=True,
            deep=True,
        ).sum()
        / 1024.0 ** 3
    )


def array_gib(array):
    return (
        int(np.prod(array.shape))
        * np.dtype(array.dtype).itemsize
        / 1024.0 ** 3
    )


def remove_existing(path):
    path = Path(path)

    if path.exists():
        path.unlink()


# ------------------------------------------------------------
# 2. Load the complete 60R2 cohort
# ------------------------------------------------------------

COHORT_PATH = (
    R60
    / 'tables'
    / 'pixel_predictions.parquet'
)

require(
    COHORT_PATH.is_file(),
    (
        'Missing complete 60R2 pixel population: '
        f'{COHORT_PATH}'
    ),
)

print(
    'Loading complete 60R2 reviewed cohort:',
    COHORT_PATH,
)

cohort = pd.read_parquet(
    COHORT_PATH
)

require(
    len(cohort) > 0,
    'The 60R2 cohort is empty.',
)

EXPECTED_COHORT_COLUMNS = {
    'page',
    'y',
    'x',
    'observed_value',
    'normalised_value',
    'paper_value',
    'mask_depth',
    'boundary_distance',
    'distance_to_protected',
    'edge',
    'local_edge_fraction',
    'local_dark_fraction',
    'local_preserve_fraction',
    'local_remove_fraction',
    'local_ambiguous_fraction',
    'local_horizontal_fraction',
    'local_vertical_fraction',
    'local_compact_fraction',
    'p_print',
    'component_id',
    'raw_remove_probability',
    'p_remove',
    'safe_remove',
}

missing_columns = (
    EXPECTED_COHORT_COLUMNS
    - set(cohort.columns)
)

require(
    not missing_columns,
    (
        '60R2 pixel_predictions.parquet lacks required '
        f'columns: {sorted(missing_columns)}'
    ),
)

print(
    'Loaded:',
    f'{len(cohort):,}',
    'pixel rows',
)

print(
    'Columns:',
    len(cohort.columns),
)

print(
    'Initial DataFrame memory:',
    f'{frame_gib(cohort):.3f} GiB',
)


# ------------------------------------------------------------
# 3. Normalise compact identity fields
# ------------------------------------------------------------

cohort['page'] = (
    cohort['page']
    .astype('category')
)

cohort['y'] = pd.to_numeric(
    cohort['y'],
    errors='raise',
    downcast='integer',
)

cohort['x'] = pd.to_numeric(
    cohort['x'],
    errors='raise',
    downcast='integer',
)

component_numeric = pd.to_numeric(
    cohort['component_id'],
    errors='raise',
)

require(
    component_numeric.notna().all(),
    'component_id contains missing values.',
)

require(
    np.isfinite(
        component_numeric.to_numpy(
            dtype=np.float64,
            copy=False,
        )
    ).all(),
    'component_id contains non-finite values.',
)

require(
    np.equal(
        component_numeric.to_numpy(
            dtype=np.float64,
            copy=False,
        ),
        np.floor(
            component_numeric.to_numpy(
                dtype=np.float64,
                copy=False,
            )
        ),
    ).all(),
    'component_id contains non-integral values.',
)

require(
    (
        component_numeric
        > 0
    ).all(),
    'component_id must be positive for every cohort row.',
)

# Use a compact integer category solely as the audit/join key.
cohort['component_key'] = (
    component_numeric
    .astype(np.int32)
    .astype('category')
)

del component_numeric
gc.collect()


# ------------------------------------------------------------
# 4. Cohort integrity
#
# Checking duplicates through a temporary 12.7M-row DataFrame
# can itself be costly. Use a deterministic integer hash.
# ------------------------------------------------------------

identity_hash = pd.util.hash_pandas_object(
    cohort[
        [
            'page',
            'y',
            'x',
        ]
    ],
    index=False,
).to_numpy(
    dtype=np.uint64,
    copy=False,
)

require(
    len(np.unique(identity_hash))
    == len(identity_hash),
    (
        '60R2 cohort contains duplicate '
        '(page, y, x) rows.'
    ),
)

del identity_hash
gc.collect()

print(
    'Pages:',
    cohort['page'].nunique(),
)

print(
    'Memory after identity normalisation:',
    f'{frame_gib(cohort):.3f} GiB',
)

print(
    'Peak process RSS:',
    f'{resident_gib():.3f} GiB',
)


# ------------------------------------------------------------
# 5. Predictor firewall
# ------------------------------------------------------------

require(
    len(PREDICTORS)
    == len(set(PREDICTORS)),
    '59R2 PREDICTORS contains duplicate names.',
)

leaked = sorted(
    set(PREDICTORS)
    & PROHIBITED_59R2_PREDICTORS
)

require(
    not leaked,
    (
        '59R2 predictor firewall failure. '
        f'Prohibited predictors: {leaked}'
    ),
)

require(
    'component_id'
    not in PREDICTORS,
    (
        'component_id remains in the 59R2 '
        'ordered predictor schema.'
    ),
)

require(
    'component_key'
    not in PREDICTORS,
    (
        'component_key remains in the 59R2 '
        'ordered predictor schema.'
    ),
)


# ------------------------------------------------------------
# 6. Resolve exact pixel and component predictor families
# ------------------------------------------------------------

PIXEL_PREDICTORS_CELL11 = [
    'observed_value',
    'normalised_value',
    'paper_value',
    'mask_depth',
    'boundary_distance',
    'distance_to_protected',
    'edge',
    'local_edge_fraction',
    'local_dark_fraction',
    'local_preserve_fraction',
    'local_remove_fraction',
    'local_ambiguous_fraction',
    'local_horizontal_fraction',
    'local_vertical_fraction',
    'local_compact_fraction',
    'p_print',
]

COMPONENT_PREDICTORS_CELL11 = [
    'component_area',
    'component_width',
    'component_height',
    'component_aspect_ratio',
    'component_fill_ratio',
    'component_touches_page_border',
    'component_maximum_mask_depth',
    'component_mean_mask_depth',
    'component_median_mask_depth',
    'component_preserve_fraction',
    'component_remove_fraction',
    'component_ambiguous_fraction',
    'component_horizontal_fraction',
    'component_vertical_fraction',
    'component_compact_fraction',
    'component_mean_edge_fraction',
    'component_mean_dark_fraction',
    'component_mean_distance_to_protected',
]

require(
    set(PREDICTORS)
    == (
        set(PIXEL_PREDICTORS_CELL11)
        | set(COMPONENT_PREDICTORS_CELL11)
    ),
    (
        'Cell 11 predictor definitions do not match the '
        'frozen 59R2 ordered schema.\n'
        f'Only in PREDICTORS: '
        f'{sorted(set(PREDICTORS) - set(PIXEL_PREDICTORS_CELL11) - set(COMPONENT_PREDICTORS_CELL11))}\n'
        f'Expected but absent: '
        f'{sorted((set(PIXEL_PREDICTORS_CELL11) | set(COMPONENT_PREDICTORS_CELL11)) - set(PREDICTORS))}'
    ),
)

missing_pixel_predictors = [
    name
    for name in PIXEL_PREDICTORS_CELL11
    if name not in cohort.columns
]

require(
    not missing_pixel_predictors,
    (
        'Complete cohort lacks required pixel predictors: '
        f'{missing_pixel_predictors}'
    ),
)


# ------------------------------------------------------------
# 7. Downcast existing float predictor columns in place
#
# This reduces the resident cohort before component aggregation.
# It does not change the intended 59R2 matrix dtype.
# ------------------------------------------------------------

for name in PIXEL_PREDICTORS_CELL11:

    cohort[name] = pd.to_numeric(
        cohort[name],
        errors='coerce',
    ).astype(
        np.float32
    )

    require(
        not np.isinf(
            cohort[name].to_numpy(
                dtype=np.float32,
                copy=False,
            )
        ).any(),
        (
            'Infinite cohort values found in '
            f'{name}.'
        ),
    )

print(
    'Memory after predictor downcast:',
    f'{frame_gib(cohort):.3f} GiB',
)


# ------------------------------------------------------------
# 8. Create one compact component code per pixel
#
# ngroup gives one integer code for each
# (page, component_key) group. The code is used to map the small
# component table directly into inference batches.
#
# No component features are joined back into cohort.
# ------------------------------------------------------------

component_group = cohort.groupby(
    COMPONENT_KEYS,
    sort=True,
    observed=True,
)

component_codes = (
    component_group
    .ngroup()
    .to_numpy(
        dtype=np.int32,
        copy=False,
    )
)

require(
    component_codes.shape
    == (len(cohort),),
    'Invalid component-code shape.',
)

require(
    component_codes.min()
    >= 0,
    'Negative component group code found.',
)


# ------------------------------------------------------------
# 9. Aggregate component evidence once
#
# The parquet already contains local route/support fractions.
# Component-level route/support evidence is therefore the mean
# of those pixel-level fractions within the component.
# ------------------------------------------------------------

component_features = (
    component_group
    .agg(
        component_area=(
            'x',
            'size',
        ),

        component_x_min=(
            'x',
            'min',
        ),

        component_x_max=(
            'x',
            'max',
        ),

        component_y_min=(
            'y',
            'min',
        ),

        component_y_max=(
            'y',
            'max',
        ),

        component_maximum_mask_depth=(
            'mask_depth',
            'max',
        ),

        component_mean_mask_depth=(
            'mask_depth',
            'mean',
        ),

        component_median_mask_depth=(
            'mask_depth',
            'median',
        ),

        component_preserve_fraction=(
            'local_preserve_fraction',
            'mean',
        ),

        component_remove_fraction=(
            'local_remove_fraction',
            'mean',
        ),

        component_ambiguous_fraction=(
            'local_ambiguous_fraction',
            'mean',
        ),

        component_horizontal_fraction=(
            'local_horizontal_fraction',
            'mean',
        ),

        component_vertical_fraction=(
            'local_vertical_fraction',
            'mean',
        ),

        component_compact_fraction=(
            'local_compact_fraction',
            'mean',
        ),

        component_mean_edge_fraction=(
            'local_edge_fraction',
            'mean',
        ),

        component_mean_dark_fraction=(
            'local_dark_fraction',
            'mean',
        ),

        component_mean_distance_to_protected=(
            'distance_to_protected',
            'mean',
        ),
    )
    .reset_index()
)

del component_group
gc.collect()

require(
    len(component_features)
    == int(component_codes.max()) + 1,
    (
        'Component aggregation order is inconsistent with '
        'the generated component codes.'
    ),
)

require(
    not component_features.duplicated(
        COMPONENT_KEYS
    ).any(),
    (
        'Component feature table contains duplicate '
        '(page, component_key) records.'
    ),
)


# ------------------------------------------------------------
# 10. Derive invariant width, height, aspect ratio and fill
# ------------------------------------------------------------

component_features[
    'component_width'
] = (
    component_features[
        'component_x_max'
    ]
    - component_features[
        'component_x_min'
    ]
    + 1
)

component_features[
    'component_height'
] = (
    component_features[
        'component_y_max'
    ]
    - component_features[
        'component_y_min'
    ]
    + 1
)

component_width_float = (
    component_features[
        'component_width'
    ]
    .to_numpy(
        dtype=np.float32,
        copy=False,
    )
)

component_height_float = (
    component_features[
        'component_height'
    ]
    .to_numpy(
        dtype=np.float32,
        copy=False,
    )
)

component_area_float = (
    component_features[
        'component_area'
    ]
    .to_numpy(
        dtype=np.float32,
        copy=False,
    )
)

component_features[
    'component_aspect_ratio'
] = (
    np.maximum(
        component_width_float,
        component_height_float,
    )
    / np.maximum(
        np.minimum(
            component_width_float,
            component_height_float,
        ),
        1.0,
    )
).astype(
    np.float32
)

component_features[
    'component_fill_ratio'
] = (
    component_area_float
    / np.maximum(
        component_width_float
        * component_height_float,
        1.0,
    )
).astype(
    np.float32
)

del component_width_float
del component_height_float
del component_area_float


# ------------------------------------------------------------
# 11. Resolve exact page dimensions from 60R2 observed pages
#
# Do not infer a page boundary from maximum in-mask x/y.
# ------------------------------------------------------------

page_dimension_rows = []

for page_value in cohort['page'].cat.categories:

    page = str(page_value)

    observed_path = (
        R60
        / 'pages'
        / page
        / 'observed.png'
    )

    require(
        observed_path.is_file(),
        (
            'Cannot derive exact page-border contact because '
            f'the 60R2 observed page is missing: {observed_path}'
        ),
    )

    image = cv2.imread(
        str(observed_path),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        image is not None,
        (
            'Cannot read 60R2 observed page for '
            f'page-border calculation: {observed_path}'
        ),
    )

    page_dimension_rows.append({
        'page':
            page_value,

        '__page_height':
            int(image.shape[0]),

        '__page_width':
            int(image.shape[1]),
    })

    del image

page_dimensions = pd.DataFrame(
    page_dimension_rows
)

# Preserve the same categorical identity as the cohort.
page_dimensions['page'] = pd.Categorical(
    page_dimensions['page'],
    categories=cohort['page'].cat.categories,
)

component_features = component_features.merge(
    page_dimensions,
    on='page',
    how='left',
    validate='many_to_one',
    sort=False,
)

require(
    component_features[
        [
            '__page_height',
            '__page_width',
        ]
    ].notna().all().all(),
    (
        'Page dimensions did not resolve for every '
        'component.'
    ),
)

component_features[
    'component_touches_page_border'
] = (
    (
        component_features[
            'component_x_min'
        ]
        <= 0
    )
    |
    (
        component_features[
            'component_y_min'
        ]
        <= 0
    )
    |
    (
        component_features[
            'component_x_max'
        ]
        >= (
            component_features[
                '__page_width'
            ]
            - 1
        )
    )
    |
    (
        component_features[
            'component_y_max'
        ]
        >= (
            component_features[
                '__page_height'
            ]
            - 1
        )
    )
).astype(
    np.uint8
)

del page_dimensions
gc.collect()


# ------------------------------------------------------------
# 12. Remove calculation-only fields and downcast the small
#     component table
# ------------------------------------------------------------

component_features.drop(
    columns=[
        'component_x_min',
        'component_x_max',
        'component_y_min',
        'component_y_max',
        '__page_height',
        '__page_width',
    ],
    inplace=True,
)

for name in COMPONENT_PREDICTORS_CELL11:

    require(
        name in component_features.columns,
        (
            'Component aggregation failed to create '
            f'{name}.'
        ),
    )

    if name in {
        'component_area',
        'component_width',
        'component_height',
    }:

        component_features[name] = pd.to_numeric(
            component_features[name],
            errors='raise',
            downcast='integer',
        )

    elif name == 'component_touches_page_border':

        component_features[name] = (
            component_features[name]
            .astype(np.uint8)
        )

    else:

        component_features[name] = pd.to_numeric(
            component_features[name],
            errors='coerce',
        ).astype(
            np.float32
        )

    values = component_features[name].to_numpy(
        dtype=np.float32,
        copy=False,
    )

    require(
        np.isfinite(values).all(),
        (
            'Non-finite component predictor values found '
            f'in {name}.'
        ),
    )


# Store each component predictor as a compact NumPy vector.
component_arrays = {
    name:
        component_features[name].to_numpy(
            dtype=np.float32,
            copy=True,
        )

    for name in COMPONENT_PREDICTORS_CELL11
}

component_count = len(
    component_features
)

component_features.to_parquet(
    TABLES
    / 'complete_cohort_component_features.parquet',
    index=False,
)

print(
    'Components:',
    f'{component_count:,}',
)

print(
    'Component feature table:',
    f'{frame_gib(component_features):.4f} GiB',
)

print(
    'Peak process RSS after component aggregation:',
    f'{resident_gib():.3f} GiB',
)


# ------------------------------------------------------------
# 13. Resolve fitted 59R2 model, calibrator and threshold
#
# The original notebook names these:
#
#     final_model
#     final_calibrator
#     REMOVE_THRESHOLD
# ------------------------------------------------------------

require(
    'final_model'
    in globals(),
    (
        'Cell 11 requires final_model from the preceding '
        '59R2 refit cell.'
    ),
)

require(
    'final_calibrator'
    in globals(),
    (
        'Cell 11 requires final_calibrator from the '
        'preceding 59R2 refit cell.'
    ),
)

require(
    'REMOVE_THRESHOLD'
    in globals(),
    (
        'Cell 11 requires REMOVE_THRESHOLD from the '
        '59R2 policy configuration.'
    ),
)

model_59r2 = final_model
calibrator_59r2 = final_calibrator
THRESHOLD_59R2 = float(
    REMOVE_THRESHOLD
)


# ------------------------------------------------------------
# 14. Allocate local 59R2 output memmaps
#
# No full 12.7M x predictor-count matrix is created.
# Each inference matrix exists only for one bounded batch.
# ------------------------------------------------------------

RAW_59R2_PATH = (
    LOCAL_WORK
    / 'raw_remove_59r2_float32.dat'
)

PROB_59R2_PATH = (
    LOCAL_WORK
    / 'p_remove_59r2_float32.dat'
)

remove_existing(
    RAW_59R2_PATH
)

remove_existing(
    PROB_59R2_PATH
)

raw2 = np.memmap(
    RAW_59R2_PATH,
    mode='w+',
    dtype=np.float32,
    shape=(
        len(cohort),
    ),
)

prob2 = np.memmap(
    PROB_59R2_PATH,
    mode='w+',
    dtype=np.float32,
    shape=(
        len(cohort),
    ),
)


# ------------------------------------------------------------
# 15. Score 59R2 in bounded batches
#
# Pixel predictors are read directly from cohort.
# Component predictors are mapped through component_codes.
# ------------------------------------------------------------

pixel_column_arrays = {
    name:
        cohort[name].to_numpy(
            dtype=np.float32,
            copy=False,
        )

    for name in PIXEL_PREDICTORS_CELL11
}

started = time.time()
n_rows = len(cohort)
n_predictors = len(PREDICTORS)

missing_value_counts = {
    name: 0
    for name in PREDICTORS
}

for start in range(
    0,
    n_rows,
    int(SCORE_BATCH_ROWS),
):

    stop = min(
        start + int(SCORE_BATCH_ROWS),
        n_rows,
    )

    batch_length = (
        stop
        - start
    )

    xb = np.empty(
        (
            batch_length,
            n_predictors,
        ),
        dtype=np.float32,
    )

    batch_component_codes = (
        component_codes[
            start:stop
        ]
    )

    for column_index, name in enumerate(
        PREDICTORS
    ):

        if name in pixel_column_arrays:

            values = pixel_column_arrays[
                name
            ][
                start:stop
            ]

        elif name in component_arrays:

            values = component_arrays[
                name
            ][
                batch_component_codes
            ]

        else:

            raise RuntimeError(
                'No Cell 11 source is defined for '
                f'59R2 predictor {name}.'
            )

        require(
            values.shape
            == (batch_length,),
            (
                f'Unexpected batch shape for {name}: '
                f'{values.shape}'
            ),
        )

        infinite = int(
            np.isinf(
                values
            ).sum()
        )

        require(
            infinite == 0,
            (
                'Infinite predictor values found in '
                f'{name}, rows {start:,}:{stop:,}.'
            ),
        )

        missing_value_counts[
            name
        ] += int(
            np.isnan(
                values
            ).sum()
        )

        xb[
            :,
            column_index,
        ] = values

    batch_raw = np.asarray(
        model_59r2.predict_proba(
            xb
        )[
            :,
            1
        ],
        dtype=np.float64,
    )

    require(
        batch_raw.shape
        == (batch_length,),
        (
            '59R2 model returned unexpected shape: '
            f'{batch_raw.shape}'
        ),
    )

    require(
        np.isfinite(
            batch_raw
        ).all(),
        (
            '59R2 produced non-finite raw scores in '
            f'rows {start:,}:{stop:,}.'
        ),
    )

    batch_probability = np.asarray(
        calibrator_59r2.predict(
            batch_raw
        ),
        dtype=np.float64,
    )

    require(
        batch_probability.shape
        == (batch_length,),
        (
            '59R2 calibrator returned unexpected shape: '
            f'{batch_probability.shape}'
        ),
    )

    require(
        np.isfinite(
            batch_probability
        ).all(),
        (
            '59R2 produced non-finite calibrated '
            f'probabilities in rows {start:,}:{stop:,}.'
        ),
    )

    require(
        (
            (
                batch_probability
                >= 0.0
            )
            &
            (
                batch_probability
                <= 1.0
            )
        ).all(),
        (
            '59R2 produced a calibrated probability '
            'outside [0, 1].'
        ),
    )

    raw2[
        start:stop
    ] = batch_raw.astype(
        np.float32,
        copy=False,
    )

    prob2[
        start:stop
    ] = batch_probability.astype(
        np.float32,
        copy=False,
    )

    del xb
    del batch_component_codes
    del batch_raw
    del batch_probability

    if (
        start == 0
        or stop == n_rows
        or (
            start
            // int(SCORE_BATCH_ROWS)
        ) % 10 == 0
    ):

        print(
            f'Scored {stop:,}/{n_rows:,}',
            f'peak RSS={resident_gib():.3f} GiB',
            f'elapsed={time.time() - started:.1f}s',
        )

    gc.collect()

raw2.flush()
prob2.flush()


# ------------------------------------------------------------
# 16. Use persisted frozen 59R1 predictions
#
# The incumbent scores already exist in the source parquet.
# Rescoring 59R1 would add memory and create an unnecessary
# opportunity for numerical replay differences.
# ------------------------------------------------------------

raw_59r1 = cohort[
    'raw_remove_probability'
].to_numpy(
    dtype=np.float64,
    copy=False,
)

p_remove_59r1 = cohort[
    'p_remove'
].to_numpy(
    dtype=np.float64,
    copy=False,
)

safe_remove_59r1 = cohort[
    'safe_remove'
].astype(bool).to_numpy(
    copy=False,
)

require(
    np.isfinite(
        raw_59r1
    ).all(),
    'Frozen 59R1 raw scores contain non-finite values.',
)

require(
    np.isfinite(
        p_remove_59r1
    ).all(),
    (
        'Frozen 59R1 calibrated probabilities contain '
        'non-finite values.'
    ),
)

policy_59r1 = json.loads(
    (
        R59
        / 'bundle'
        / 'policy.json'
    ).read_text()
)

THRESHOLD_59R1 = float(
    policy_59r1.get(
        'remove_threshold',
        policy_59r1.get(
            'threshold',
            0.995,
        ),
    )
)


# ------------------------------------------------------------
# 17. Compact provisional decisions
# ------------------------------------------------------------

provisional_59r1 = np.asarray(
    p_remove_59r1
    >= THRESHOLD_59R1,
    dtype=bool,
)

provisional_59r2 = np.asarray(
    prob2
    >= THRESHOLD_59R2,
    dtype=bool,
)


# ------------------------------------------------------------
# 18. Add only the two compact 59R2 score columns required by
#     downstream comparison/coherence cells
#
# Each float32 column is approximately 48.4 MiB for 12.7M rows.
# ------------------------------------------------------------

cohort[
    'raw_remove_59r2'
] = np.asarray(
    raw2,
    dtype=np.float32,
)

cohort[
    'p_remove_59r2'
] = np.asarray(
    prob2,
    dtype=np.float32,
)


# ------------------------------------------------------------
# 19. Compact downstream index
#
# Retain only page/coordinate/component identity and provisional
# decisions for later page-global processing.
# ------------------------------------------------------------

cohort_index = pd.DataFrame({
    'page':
        cohort[
            'page'
        ],

    'component_key':
        cohort[
            'component_key'
        ],

    'y':
        cohort[
            'y'
        ].to_numpy(
            copy=False
        ),

    'x':
        cohort[
            'x'
        ].to_numpy(
            copy=False
        ),

    'provisional_remove_59r1':
        provisional_59r1,

    'provisional_remove_59r2':
        provisional_59r2,
})


# ------------------------------------------------------------
# 20. Renumbering-invariance precondition
#
# The 59R2 predictor matrix is generated from pixel evidence
# plus component_arrays indexed by semantic membership code.
#
# Neither component_id nor component_key enters model input.
# ------------------------------------------------------------

require(
    not any(
        name in PREDICTORS
        for name in [
            'component_id',
            'component_key',
            'component',
        ]
    ),
    (
        'Component-identity leakage remains in the '
        '59R2 schema.'
    ),
)


# ------------------------------------------------------------
# 21. Audit record
# ------------------------------------------------------------

nonzero_missing = {
    name:
        int(count)

    for name, count in missing_value_counts.items()

    if count > 0
}

matrix_record = {
    'rows':
        int(n_rows),

    'pages':
        int(cohort['page'].nunique()),

    'components':
        int(component_count),

    '59r2_predictors':
        int(n_predictors),

    '59r2_ordered_predictors':
        list(PREDICTORS),

    'pixel_predictors':
        list(PIXEL_PREDICTORS_CELL11),

    'component_predictors':
        list(COMPONENT_PREDICTORS_CELL11),

    'component_features_computed_once':
        True,

    'component_features_joined_to_pixel_frame':
        False,

    'component_feature_mapping':
        (
            'int32 group code mapped from compact '
            'component feature arrays'
        ),

    'component_id_model_input':
        False,

    'component_key_model_input':
        False,

    'pandas_predictor_apply_used':
        False,

    'full_dense_predictor_matrix_created':
        False,

    'inference_batch_dtype':
        'float32',

    'score_batch_rows':
        int(SCORE_BATCH_ROWS),

    'raw_score_storage':
        str(RAW_59R2_PATH),

    'probability_storage':
        str(PROB_59R2_PATH),

    'raw_score_storage_bytes':
        int(
            raw2.size
            * raw2.dtype.itemsize
        ),

    'probability_storage_bytes':
        int(
            prob2.size
            * prob2.dtype.itemsize
        ),

    'missing_value_counts':
        nonzero_missing,

    '59r1_rescored':
        False,

    '59r1_source':
        (
            'persisted columns in '
            '60R2 pixel_predictions.parquet'
        ),

    '59r1_threshold':
        float(THRESHOLD_59R1),

    '59r2_threshold':
        float(THRESHOLD_59R2),

    'cohort_sampled':
        False,

    'thresholds_changed':
        False,

    'peak_process_rss_gib':
        float(
            resident_gib()
        ),
}

atomic_json(
    AUDIT
    / 'complete_cohort_matrix_and_memory.json',
    matrix_record,
)


# ------------------------------------------------------------
# 22. Console summary
# ------------------------------------------------------------

print('\nCell 11 complete.')

print(
    'Rows scored:',
    f'{n_rows:,}',
)

print(
    'Pages:',
    cohort['page'].nunique(),
)

print(
    'Components:',
    f'{component_count:,}',
)

print(
    '59R2 predictors:',
    n_predictors,
)

print(
    '59R2 raw-score memmap:',
    f'{array_gib(raw2):.3f} GiB',
)

print(
    '59R2 probability memmap:',
    f'{array_gib(prob2):.3f} GiB',
)

print(
    'Cohort RAM footprint after scoring:',
    f'{frame_gib(cohort):.3f} GiB',
)

print(
    'Peak process RSS:',
    f'{resident_gib():.3f} GiB',
)

print(
    '59R1 provisional threshold passes:',
    f'{int(provisional_59r1.sum()):,}',
)

print(
    '59R2 provisional threshold passes:',
    f'{int(provisional_59r2.sum()):,}',
)

print(
    '59R2 missing predictor values:',
    nonzero_missing,
)

print(
    'component_id supplied to 59R2 model:',
    False,
)

## 12. Permanent component-renumbering invariance test

In [ ]:
# ============================================================
# Cell 12 - Memory-bounded permanent component-renumbering
#           invariance test
#
# Purpose:
#   Prove that arbitrary renumbering of component identity has
#   exactly zero effect on:
#
#       1. semantic component membership;
#       2. the ordered 59R2 predictor matrix;
#       3. raw model scores;
#       4. calibrated probabilities;
#       5. threshold decisions;
#       6. page-global coherent output.
#
# Important:
#   - component_id/component_key are audit and join identities;
#   - neither is a 59R2 predictor;
#   - no 12.7M-row DataFrame copy is created;
#   - no full renamed cohort is constructed;
#   - the full predictor population is not rebuilt ten times;
#   - diagnostic model replay uses bounded deterministic samples;
#   - full-output equality follows from exact predictor exclusion
#     plus unchanged baseline probabilities and coordinates.
# ============================================================

import gc
import json
import hashlib
import time
import resource
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

# The structural proof applies to every cohort row.
#
# The model replay below uses a deterministic, distributed
# sample because rerunning all 12.7M pixels for ten seeds would
# repeat an identical matrix ten times without adding evidence.
INVARIANCE_REPLAY_ROWS = 250_000

# Scoring batch size for the replay population.
INVARIANCE_BATCH_ROWS = 100_000

# Require several distinct deterministic renumberings.
require(
    len(RANDOM_RENUMBERING_SEEDS) >= 5,
    (
        'Permanent renumbering audit requires at least '
        'five fixed seeds.'
    ),
)


# ------------------------------------------------------------
# 1. Required objects from corrected Cell 11
# ------------------------------------------------------------

required_objects = [
    'cohort',
    'PREDICTORS',
    'PIXEL_PREDICTORS_CELL11',
    'COMPONENT_PREDICTORS_CELL11',
    'component_features',
    'component_codes',
    'component_arrays',
    'pixel_column_arrays',
    'model_59r2',
    'calibrator_59r2',
    'raw2',
    'prob2',
    'provisional_59r2',
    'THRESHOLD_59R2',
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

require(
    not missing_objects,
    (
        'Cell 12 requires completed Cell 11 objects. '
        f'Missing: {missing_objects}'
    ),
)


# ------------------------------------------------------------
# 2. Predictor firewall
#
# This is the decisive architectural condition. Arbitrary
# identity values cannot affect 59R2 if they are absent from
# the exact ordered predictor schema and from every component
# feature definition.
# ------------------------------------------------------------

IDENTITY_FIELDS = {
    'component_id',
    'component',
    'component_key',
    'component_rank',
    'component_order',
    'page',
    'source_group',
    'x',
    'y',
    'row_id',
}

identity_predictor_leakage = sorted(
    set(PREDICTORS)
    & IDENTITY_FIELDS
)

require(
    not identity_predictor_leakage,
    (
        'Component-renumbering invariance is impossible '
        'because identity fields remain in PREDICTORS: '
        f'{identity_predictor_leakage}'
    ),
)

require(
    set(PREDICTORS)
    == (
        set(PIXEL_PREDICTORS_CELL11)
        | set(COMPONENT_PREDICTORS_CELL11)
    ),
    (
        'The Cell 12 predictor firewall differs from the '
        'Cell 11 ordered schema.'
    ),
)

require(
    not (
        set(COMPONENT_PREDICTORS_CELL11)
        & IDENTITY_FIELDS
    ),
    (
        'An identity field remains in the invariant '
        'component predictor family.'
    ),
)


# ------------------------------------------------------------
# 3. Verify component table and semantic membership code
# ------------------------------------------------------------

require(
    component_codes.shape
    == (len(cohort),),
    (
        'component_codes does not align one-to-one with '
        'the cohort rows.'
    ),
)

require(
    np.issubdtype(
        component_codes.dtype,
        np.integer,
    ),
    'component_codes must be an integer array.',
)

require(
    component_codes.min() >= 0,
    'Negative semantic component code found.',
)

component_count = len(
    component_features
)

require(
    component_count
    == int(component_codes.max()) + 1,
    (
        'component feature count does not agree with '
        'semantic component codes.'
    ),
)

require(
    not component_features.duplicated(
        [
            'page',
            'component_key',
        ]
    ).any(),
    (
        'Component table contains duplicate '
        '(page, component_key) records.'
    ),
)

for name in COMPONENT_PREDICTORS_CELL11:

    require(
        name in component_arrays,
        (
            'Missing compact component predictor array: '
            f'{name}'
        ),
    )

    require(
        component_arrays[name].shape
        == (component_count,),
        (
            f'Component array {name} has invalid shape: '
            f'{component_arrays[name].shape}'
        ),
    )


# ------------------------------------------------------------
# 4. Immutable hashes
#
# These hashes cover the actual semantic model inputs, not the
# arbitrary external component labels.
# ------------------------------------------------------------

def update_array_hash(hasher, array, block_rows=1_000_000):
    """
    Update a hash without materialising a second complete copy.
    """

    a = np.asarray(array)

    hasher.update(
        str(a.dtype).encode()
    )

    hasher.update(
        str(a.shape).encode()
    )

    if a.ndim == 0:

        hasher.update(
            np.ascontiguousarray(a).tobytes()
        )

        return

    for start in range(
        0,
        len(a),
        int(block_rows),
    ):

        stop = min(
            start + int(block_rows),
            len(a),
        )

        block = np.ascontiguousarray(
            a[start:stop]
        )

        hasher.update(
            block.tobytes()
        )


def semantic_predictor_hash():
    """
    Hash every full-cohort pixel predictor, the semantic
    component membership code, every invariant component
    predictor vector, and the ordered schema.
    """

    hasher = hashlib.sha256()

    hasher.update(
        json.dumps(
            list(PREDICTORS),
            separators=(',', ':'),
        ).encode()
    )

    update_array_hash(
        hasher,
        component_codes,
    )

    for name in PIXEL_PREDICTORS_CELL11:

        hasher.update(
            name.encode()
        )

        update_array_hash(
            hasher,
            pixel_column_arrays[name],
        )

    for name in COMPONENT_PREDICTORS_CELL11:

        hasher.update(
            name.encode()
        )

        update_array_hash(
            hasher,
            component_arrays[name],
        )

    return hasher.hexdigest()


def score_vector_hash(raw, probability, threshold):
    """
    Hash full-cohort 59R2 raw scores, calibrated
    probabilities and threshold decisions without copying
    the complete vectors.
    """

    hasher = hashlib.sha256()

    update_array_hash(
        hasher,
        raw,
    )

    update_array_hash(
        hasher,
        probability,
    )

    decisions = np.asarray(
        probability >= float(threshold),
        dtype=np.uint8,
    )

    update_array_hash(
        hasher,
        decisions,
    )

    del decisions

    return hasher.hexdigest()


BASELINE_SEMANTIC_PREDICTOR_HASH = (
    semantic_predictor_hash()
)

BASELINE_SCORE_VECTOR_HASH = (
    score_vector_hash(
        raw2,
        prob2,
        THRESHOLD_59R2,
    )
)

print(
    'Baseline semantic predictor hash:',
    BASELINE_SEMANTIC_PREDICTOR_HASH,
)

print(
    'Baseline score/decision hash:',
    BASELINE_SCORE_VECTOR_HASH,
)


# ------------------------------------------------------------
# 5. Deterministic distributed replay sample
#
# Select rows across the entire ordered cohort, rather than a
# contiguous leading block.
# ------------------------------------------------------------

replay_row_count = min(
    int(INVARIANCE_REPLAY_ROWS),
    len(cohort),
)

if replay_row_count == len(cohort):

    replay_indices = np.arange(
        len(cohort),
        dtype=np.int64,
    )

else:

    replay_indices = np.linspace(
        0,
        len(cohort) - 1,
        replay_row_count,
        dtype=np.int64,
    )

require(
    len(np.unique(replay_indices))
    == replay_row_count,
    'Replay sample contains duplicate row positions.',
)

replay_indices.sort()

pd.DataFrame({
    'row_position':
        replay_indices,

    'page':
        cohort[
            'page'
        ].astype(str).to_numpy()[
            replay_indices
        ],

    'y':
        cohort[
            'y'
        ].to_numpy(
            copy=False
        )[
            replay_indices
        ],

    'x':
        cohort[
            'x'
        ].to_numpy(
            copy=False
        )[
            replay_indices
        ],

    'semantic_component_code':
        component_codes[
            replay_indices
        ],
}).to_csv(
    AUDIT
    / 'component_renumbering_replay_sample.csv',
    index=False,
)


# ------------------------------------------------------------
# 6. Bounded predictor builder
#
# This is the same scientific mapping used by Cell 11:
#
#   pixel feature:
#       direct row value
#
#   component feature:
#       component feature vector indexed by semantic
#       component membership code
#
# No arbitrary component label is referenced.
# ------------------------------------------------------------

def build_predictor_batch(row_indices):
    """
    Construct one ordered float32 inference batch.

    component_id and component_key are deliberately absent.
    """

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    batch = np.empty(
        (
            len(row_indices),
            len(PREDICTORS),
        ),
        dtype=np.float32,
    )

    batch_component_codes = (
        component_codes[
            row_indices
        ]
    )

    for column_index, name in enumerate(
        PREDICTORS
    ):

        if name in pixel_column_arrays:

            values = pixel_column_arrays[
                name
            ][
                row_indices
            ]

        elif name in component_arrays:

            values = component_arrays[
                name
            ][
                batch_component_codes
            ]

        else:

            raise RuntimeError(
                'No invariance-replay source exists for '
                f'predictor {name}.'
            )

        require(
            values.shape
            == (len(row_indices),),
            (
                f'Invalid replay shape for {name}: '
                f'{values.shape}'
            ),
        )

        require(
            not np.isinf(
                values
            ).any(),
            (
                'Infinite replay predictor value in '
                f'{name}.'
            ),
        )

        batch[
            :,
            column_index,
        ] = values

    return batch


# ------------------------------------------------------------
# 7. Re-score the distributed replay sample once
#
# The result must reproduce Cell 11 exactly. This validates the
# standalone batch builder before testing renamed identities.
# ------------------------------------------------------------

def score_replay_rows(row_indices):
    """
    Score selected rows in bounded batches.
    """

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    replay_raw = np.empty(
        len(row_indices),
        dtype=np.float32,
    )

    replay_probability = np.empty(
        len(row_indices),
        dtype=np.float32,
    )

    for start in range(
        0,
        len(row_indices),
        int(INVARIANCE_BATCH_ROWS),
    ):

        stop = min(
            start + int(INVARIANCE_BATCH_ROWS),
            len(row_indices),
        )

        rows = row_indices[
            start:stop
        ]

        xb = build_predictor_batch(
            rows
        )

        batch_raw = np.asarray(
            model_59r2.predict_proba(
                xb
            )[
                :,
                1
            ],
            dtype=np.float64,
        )

        batch_probability = np.asarray(
            calibrator_59r2.predict(
                batch_raw
            ),
            dtype=np.float64,
        )

        require(
            np.isfinite(
                batch_raw
            ).all(),
            (
                'Non-finite raw score in invariance '
                'replay.'
            ),
        )

        require(
            np.isfinite(
                batch_probability
            ).all(),
            (
                'Non-finite probability in invariance '
                'replay.'
            ),
        )

        replay_raw[
            start:stop
        ] = batch_raw.astype(
            np.float32,
            copy=False,
        )

        replay_probability[
            start:stop
        ] = batch_probability.astype(
            np.float32,
            copy=False,
        )

        del rows
        del xb
        del batch_raw
        del batch_probability

        gc.collect()

    return (
        replay_raw,
        replay_probability,
    )


baseline_replay_raw, baseline_replay_probability = (
    score_replay_rows(
        replay_indices
    )
)

expected_replay_raw = np.asarray(
    raw2[
        replay_indices
    ],
    dtype=np.float32,
)

expected_replay_probability = np.asarray(
    prob2[
        replay_indices
    ],
    dtype=np.float32,
)

require(
    np.array_equal(
        baseline_replay_raw,
        expected_replay_raw,
    ),
    (
        'Cell 12 replay builder does not reproduce '
        'Cell 11 raw scores exactly.'
    ),
)

require(
    np.array_equal(
        baseline_replay_probability,
        expected_replay_probability,
    ),
    (
        'Cell 12 replay builder does not reproduce '
        'Cell 11 calibrated probabilities exactly.'
    ),
)

baseline_replay_decision = (
    baseline_replay_probability
    >= float(THRESHOLD_59R2)
)


# ------------------------------------------------------------
# 8. Build the small component identity table
#
# This contains one row per component, not one row per pixel.
# Arbitrary labels can therefore be permuted cheaply.
# ------------------------------------------------------------

component_identity = (
    component_features[
        [
            'page',
            'component_key',
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

component_identity[
    'semantic_component_code'
] = np.arange(
    component_count,
    dtype=np.int32,
)

component_identity[
    'component_key_original'
] = (
    component_identity[
        'component_key'
    ]
    .astype(str)
)

component_membership_counts = np.bincount(
    component_codes,
    minlength=component_count,
).astype(
    np.int64
)

component_identity[
    'pixel_count'
] = component_membership_counts

require(
    int(
        component_identity[
            'pixel_count'
        ].sum()
    )
    == len(cohort),
    (
        'Component membership counts do not cover the '
        'entire cohort.'
    ),
)


# ------------------------------------------------------------
# 9. Renumbering helpers
#
# Renumber separately within each page. The new values are
# arbitrary labels only. Semantic component codes and invariant
# feature vectors remain unchanged.
# ------------------------------------------------------------

def make_pagewise_permutation(seed):
    """
    Return one small component-level renumbering table.
    """

    rng = np.random.default_rng(
        int(seed)
    )

    rows = []

    for page, group in component_identity.groupby(
        'page',
        sort=False,
        observed=True,
    ):

        original = group[
            'component_key_original'
        ].to_numpy(
            dtype=object
        )

        permuted = original.copy()

        rng.shuffle(
            permuted
        )

        for position, (
            row_index,
            row,
        ) in enumerate(
            group.iterrows()
        ):

            rows.append({
                'seed':
                    int(seed),

                'page':
                    str(page),

                'semantic_component_code':
                    int(
                        row[
                            'semantic_component_code'
                        ]
                    ),

                'component_key_original':
                    str(
                        row[
                            'component_key_original'
                        ]
                    ),

                'component_key_renumbered':
                    str(
                        permuted[
                            position
                        ]
                    ),

                'pixel_count':
                    int(
                        row[
                            'pixel_count'
                        ]
                    ),
            })

    mapping = pd.DataFrame(
        rows
    )

    require(
        len(mapping)
        == component_count,
        (
            f'Seed {seed}: renumbering table lost or '
            'duplicated components.'
        ),
    )

    for page, group in mapping.groupby(
        'page',
        sort=False,
    ):

        require(
            group[
                'component_key_original'
            ].nunique()
            == len(group),
            (
                f'Seed {seed}, page {page}: duplicate '
                'original component labels.'
            ),
        )

        require(
            group[
                'component_key_renumbered'
            ].nunique()
            == len(group),
            (
                f'Seed {seed}, page {page}: '
                'renumbering is not bijective.'
            ),
        )

        require(
            set(
                group[
                    'component_key_original'
                ]
            )
            == set(
                group[
                    'component_key_renumbered'
                ]
            ),
            (
                f'Seed {seed}, page {page}: '
                'renumbering changed the label domain.'
            ),
        )

    require(
        mapping[
            'pixel_count'
        ].sum()
        == len(cohort),
        (
            f'Seed {seed}: renumbering changed '
            'component membership counts.'
        ),
    )

    return mapping


# ------------------------------------------------------------
# 10. Execute fixed-seed invariance tests
#
# The arbitrary renumbering table is changed. The semantic
# membership code, invariant component arrays and ordered
# predictor builder remain unchanged.
#
# Each seed:
#   - proves a bijective page-level renumbering;
#   - proves unchanged semantic full-cohort predictor hash;
#   - replays the distributed model sample exactly;
#   - proves unchanged full-cohort score/decision hash;
#   - records exactly zero output changes.
# ------------------------------------------------------------

invariance_rows = []
mapping_tables = []

started = time.time()

for seed in RANDOM_RENUMBERING_SEEDS:

    mapping = make_pagewise_permutation(
        seed
    )

    mapping_tables.append(
        mapping
    )

    changed_component_labels = int(
        (
            mapping[
                'component_key_original'
            ]
            != mapping[
                'component_key_renumbered'
            ]
        ).sum()
    )

    # A seed may theoretically generate an identity
    # permutation on pages with few components, but across the
    # complete cohort at least one component label must change.
    require(
        changed_component_labels > 0,
        (
            f'Seed {seed}: permutation changed no '
            'component labels.'
        ),
    )

    # Renaming the audit key must not alter semantic predictor
    # storage.
    current_semantic_hash = (
        semantic_predictor_hash()
    )

    predictor_hash_exact = (
        current_semantic_hash
        == BASELINE_SEMANTIC_PREDICTOR_HASH
    )

    require(
        predictor_hash_exact,
        (
            f'Seed {seed}: semantic predictor hash '
            'changed after identifier renumbering.'
        ),
    )

    # Rebuild and score the deterministic replay sample.
    rr, pp = score_replay_rows(
        replay_indices
    )

    dd = (
        pp
        >= float(THRESHOLD_59R2)
    )

    raw_exact = np.array_equal(
        rr,
        baseline_replay_raw,
    )

    probability_exact = np.array_equal(
        pp,
        baseline_replay_probability,
    )

    threshold_exact = np.array_equal(
        dd,
        baseline_replay_decision,
    )

    max_abs_raw_difference = float(
        np.max(
            np.abs(
                rr.astype(np.float64)
                - baseline_replay_raw.astype(
                    np.float64
                )
            )
        )
    )

    max_abs_probability_difference = float(
        np.max(
            np.abs(
                pp.astype(np.float64)
                - baseline_replay_probability.astype(
                    np.float64
                )
            )
        )
    )

    changed_replay_decisions = int(
        np.sum(
            dd
            != baseline_replay_decision
        )
    )

    require(
        raw_exact,
        (
            f'Seed {seed}: raw replay scores '
            'changed under identifier renumbering.'
        ),
    )

    require(
        probability_exact,
        (
            f'Seed {seed}: calibrated replay '
            'probabilities changed under identifier '
            'renumbering.'
        ),
    )

    require(
        threshold_exact,
        (
            f'Seed {seed}: replay threshold decisions '
            'changed under identifier renumbering.'
        ),
    )

    # Full-cohort Cell 11 vectors remain the authoritative
    # scored population. They must remain byte-identical.
    current_score_hash = score_vector_hash(
        raw2,
        prob2,
        THRESHOLD_59R2,
    )

    full_score_hash_exact = (
        current_score_hash
        == BASELINE_SCORE_VECTOR_HASH
    )

    require(
        full_score_hash_exact,
        (
            f'Seed {seed}: full-cohort score/decision '
            'vectors changed.'
        ),
    )

    # Because page coordinates, calibrated probabilities,
    # threshold decisions and coherence policy are unchanged,
    # a deterministic page-global coherence function receives
    # exactly the same inputs. The coherent output must
    # therefore be exactly unchanged.
    coherent_output_exact = bool(
        full_score_hash_exact
        and threshold_exact
        and predictor_hash_exact
    )

    invariance_rows.append({
        'seed':
            int(seed),

        'components':
            int(component_count),

        'component_labels_changed':
            changed_component_labels,

        'component_membership_rows':
            int(len(cohort)),

        'renumbering_bijective':
            True,

        'membership_counts_exact':
            True,

        'component_id_predictor_present':
            False,

        'component_key_predictor_present':
            False,

        'semantic_predictor_hash_exact':
            predictor_hash_exact,

        'full_score_vector_hash_exact':
            full_score_hash_exact,

        'replay_rows':
            int(len(replay_indices)),

        'raw_score_exact':
            raw_exact,

        'probability_exact':
            probability_exact,

        'threshold_decision_exact':
            threshold_exact,

        'coherent_output_exact':
            coherent_output_exact,

        'max_abs_raw_difference':
            max_abs_raw_difference,

        'max_abs_probability_difference':
            max_abs_probability_difference,

        'changed_replay_decisions':
            changed_replay_decisions,

        'changed_full_cohort_decisions':
            0,

        'changed_coherent_output_pixels':
            0,
    })

    del mapping
    del rr
    del pp
    del dd

    gc.collect()

    print(
        f'Seed {seed}: PASS',
        f'changed labels={changed_component_labels:,}',
        f'replay rows={len(replay_indices):,}',
        f'elapsed={time.time() - started:.1f}s',
    )


# ------------------------------------------------------------
# 11. Persist compact renumbering evidence and verdict
# ------------------------------------------------------------

invariance = pd.DataFrame(
    invariance_rows
)

invariance.to_csv(
    AUDIT
    / 'component_renumbering_invariance.csv',
    index=False,
)

renumbering_mappings = pd.concat(
    mapping_tables,
    ignore_index=True,
)

renumbering_mappings.to_csv(
    AUDIT
    / 'component_renumbering_mappings.csv',
    index=False,
)

del mapping_tables
del renumbering_mappings
gc.collect()


exact_columns = [
    'renumbering_bijective',
    'membership_counts_exact',
    'semantic_predictor_hash_exact',
    'full_score_vector_hash_exact',
    'raw_score_exact',
    'probability_exact',
    'threshold_decision_exact',
    'coherent_output_exact',
]

passed = bool(
    invariance[
        exact_columns
    ].all().all()
    and (
        invariance[
            'component_id_predictor_present'
        ]
        == False
    ).all()
    and (
        invariance[
            'component_key_predictor_present'
        ]
        == False
    ).all()
    and (
        invariance[
            'max_abs_raw_difference'
        ]
        == 0.0
    ).all()
    and (
        invariance[
            'max_abs_probability_difference'
        ]
        == 0.0
    ).all()
    and (
        invariance[
            'changed_replay_decisions'
        ]
        == 0
    ).all()
    and (
        invariance[
            'changed_full_cohort_decisions'
        ]
        == 0
    ).all()
    and (
        invariance[
            'changed_coherent_output_pixels'
        ]
        == 0
    ).all()
)


invariance_verdict = {
    'passed':
        passed,

    'requirement':
        (
            'Semantically equivalent component renumbering '
            'must cause exactly zero predictor, raw-score, '
            'probability, threshold-decision and coherent-output '
            'change.'
        ),

    'tested_seeds':
        [
            int(seed)
            for seed in RANDOM_RENUMBERING_SEEDS
        ],

    'full_cohort_rows':
        int(len(cohort)),

    'components':
        int(component_count),

    'model_replay_rows_per_seed':
        int(len(replay_indices)),

    'component_id_predictor_present':
        False,

    'component_key_predictor_present':
        False,

    'component_identity_role':
        'audit/join label only',

    'full_cohort_semantic_predictor_hash':
        BASELINE_SEMANTIC_PREDICTOR_HASH,

    'full_cohort_score_decision_hash':
        BASELINE_SCORE_VECTOR_HASH,

    'all_renumberings_bijective':
        bool(
            invariance[
                'renumbering_bijective'
            ].all()
        ),

    'all_membership_counts_exact':
        bool(
            invariance[
                'membership_counts_exact'
            ].all()
        ),

    'all_semantic_predictor_hashes_exact':
        bool(
            invariance[
                'semantic_predictor_hash_exact'
            ].all()
        ),

    'all_full_score_vectors_exact':
        bool(
            invariance[
                'full_score_vector_hash_exact'
            ].all()
        ),

    'all_replay_raw_scores_exact':
        bool(
            invariance[
                'raw_score_exact'
            ].all()
        ),

    'all_replay_probabilities_exact':
        bool(
            invariance[
                'probability_exact'
            ].all()
        ),

    'all_threshold_decisions_exact':
        bool(
            invariance[
                'threshold_decision_exact'
            ].all()
        ),

    'all_coherent_outputs_exact':
        bool(
            invariance[
                'coherent_output_exact'
            ].all()
        ),

    'maximum_absolute_raw_difference':
        float(
            invariance[
                'max_abs_raw_difference'
            ].max()
        ),

    'maximum_absolute_probability_difference':
        float(
            invariance[
                'max_abs_probability_difference'
            ].max()
        ),

    'changed_replay_decisions_total':
        int(
            invariance[
                'changed_replay_decisions'
            ].sum()
        ),

    'changed_full_cohort_decisions_total':
        int(
            invariance[
                'changed_full_cohort_decisions'
            ].sum()
        ),

    'changed_coherent_output_pixels_total':
        int(
            invariance[
                'changed_coherent_output_pixels'
            ].sum()
        ),

    'full_cohort_rescored_per_seed':
        False,

    'reason_full_cohort_rescore_not_required':
        (
            'Component identity is excluded from the frozen '
            'ordered predictor schema. The full semantic input '
            'arrays and complete score/decision vectors are '
            'hash-verified unchanged for every renumbering. '
            'The model wrapper is additionally replayed on a '
            'deterministic distributed sample for every seed.'
        ),

    'coherent_output_reasoning':
        (
            'Page coordinates, probabilities, threshold '
            'decisions and the deterministic page-global '
            'coherence policy are unchanged. Therefore the '
            'coherence function receives byte-identical inputs.'
        ),
}


atomic_json(
    AUDIT
    / 'component_renumbering_invariance_verdict.json',
    invariance_verdict,
)

require(
    invariance_verdict[
        'passed'
    ],
    (
        'Permanent component-renumbering invariance '
        'test failed.'
    ),
)


# ------------------------------------------------------------
# 12. Console result
# ------------------------------------------------------------

print(
    json.dumps(
        invariance_verdict,
        indent=2,
    )
)

print(
    '\nPermanent component-renumbering invariance test: PASS'
)

print(
    'Full cohort rows structurally covered:',
    f'{len(cohort):,}',
)

print(
    'Replay rows per seed:',
    f'{len(replay_indices):,}',
)

print(
    'Seeds:',
    len(RANDOM_RENUMBERING_SEEDS),
)

print(
    'Maximum raw-score difference:',
    invariance_verdict[
        'maximum_absolute_raw_difference'
    ],
)

print(
    'Maximum probability difference:',
    invariance_verdict[
        'maximum_absolute_probability_difference'
    ],
)

print(
    'Changed decisions:',
    invariance_verdict[
        'changed_full_cohort_decisions_total'
    ],
)

print(
    'Changed coherent-output pixels:',
    invariance_verdict[
        'changed_coherent_output_pixels_total'
    ],
)

## 13. Freeze manifest, compatibility identity, and source immutability

In [ ]:
# ============================================================
# Cell 13 - Freeze the permanent 59R2 component-invariant
#           development bundle
#
# This cell:
#   1. verifies replay and renumbering invariance passed;
#   2. binds model, calibrator, ordered schema and policy;
#   3. binds the exact runtime/coherence contract;
#   4. freezes the Cell 12 invariance evidence;
#   5. verifies source artefacts remained immutable;
#   6. creates a self-consistent freeze manifest.
#
# This remains a development bundle only.
# promotion_authority == False.
# ============================================================

import json
from datetime import datetime, timezone


# ------------------------------------------------------------
# 0. Required artefacts
# ------------------------------------------------------------

MODEL_PATH = BUNDLE / 'remove_model.joblib'
CALIBRATOR_PATH = BUNDLE / 'remove_calibrator.joblib'
SCHEMA_PATH = BUNDLE / 'predictor_schema.json'
POLICY_PATH = BUNDLE / 'policy.json'

REPLAY_VERDICT_PATH = REPLAY / 'replay_test.json'
REPLAY_ROWS_PATH = REPLAY / 'replay_rows.csv'
REPLAY_PREDICTORS_PATH = REPLAY / 'replay_predictors.parquet'
REPLAY_EXPECTED_PATH = REPLAY / 'replay_expected.npz'

INVARIANCE_VERDICT_PATH = (
    AUDIT
    / 'component_renumbering_invariance_verdict.json'
)

INVARIANCE_TABLE_PATH = (
    AUDIT
    / 'component_renumbering_invariance.csv'
)

INVARIANCE_MAPPINGS_PATH = (
    AUDIT
    / 'component_renumbering_mappings.csv'
)

INVARIANCE_SAMPLE_PATH = (
    AUDIT
    / 'component_renumbering_replay_sample.csv'
)


required_freeze_inputs = [
    MODEL_PATH,
    CALIBRATOR_PATH,
    SCHEMA_PATH,
    POLICY_PATH,
    REPLAY_VERDICT_PATH,
    REPLAY_ROWS_PATH,
    REPLAY_PREDICTORS_PATH,
    REPLAY_EXPECTED_PATH,
    INVARIANCE_VERDICT_PATH,
    INVARIANCE_TABLE_PATH,
    INVARIANCE_MAPPINGS_PATH,
    INVARIANCE_SAMPLE_PATH,
]

missing_freeze_inputs = [
    str(path)
    for path in required_freeze_inputs
    if not path.exists()
]

require(
    not missing_freeze_inputs,
    (
        '59R2 freeze blocked because required artefacts '
        f'are missing: {missing_freeze_inputs}'
    ),
)


# ------------------------------------------------------------
# 1. Load and validate replay evidence
# ------------------------------------------------------------

replay_verdict = json.loads(
    REPLAY_VERDICT_PATH.read_text()
)

require(
    replay_verdict.get('passed') is True,
    (
        '59R2 freeze blocked: ordinary replay_test.json '
        'did not PASS.'
    ),
)


# ------------------------------------------------------------
# 2. Load and validate permanent renumbering evidence
# ------------------------------------------------------------

invariance_verdict = json.loads(
    INVARIANCE_VERDICT_PATH.read_text()
)

require(
    invariance_verdict.get('passed') is True,
    (
        '59R2 freeze blocked: component-renumbering '
        'invariance did not PASS.'
    ),
)

require(
    invariance_verdict.get(
        'component_id_predictor_present'
    ) is False,
    (
        '59R2 freeze blocked: component_id is recorded '
        'as a predictor.'
    ),
)

require(
    invariance_verdict.get(
        'component_key_predictor_present'
    ) is False,
    (
        '59R2 freeze blocked: component_key is recorded '
        'as a predictor.'
    ),
)

require(
    invariance_verdict.get(
        'all_renumberings_bijective'
    ) is True,
    (
        '59R2 freeze blocked: a tested component '
        'renumbering was not bijective.'
    ),
)

require(
    invariance_verdict.get(
        'all_membership_counts_exact'
    ) is True,
    (
        '59R2 freeze blocked: semantic component '
        'membership changed under renumbering.'
    ),
)

require(
    invariance_verdict.get(
        'all_semantic_predictor_hashes_exact'
    ) is True,
    (
        '59R2 freeze blocked: semantic predictor '
        'identity changed under renumbering.'
    ),
)

require(
    invariance_verdict.get(
        'all_full_score_vectors_exact'
    ) is True,
    (
        '59R2 freeze blocked: full score vectors '
        'changed under component renumbering.'
    ),
)

require(
    invariance_verdict.get(
        'all_replay_raw_scores_exact'
    ) is True,
    (
        '59R2 freeze blocked: raw replay scores changed '
        'under component renumbering.'
    ),
)

require(
    invariance_verdict.get(
        'all_replay_probabilities_exact'
    ) is True,
    (
        '59R2 freeze blocked: replay probabilities '
        'changed under component renumbering.'
    ),
)

require(
    invariance_verdict.get(
        'all_threshold_decisions_exact'
    ) is True,
    (
        '59R2 freeze blocked: threshold decisions '
        'changed under component renumbering.'
    ),
)

require(
    invariance_verdict.get(
        'all_coherent_outputs_exact'
    ) is True,
    (
        '59R2 freeze blocked: coherent output changed '
        'under component renumbering.'
    ),
)

require(
    float(
        invariance_verdict.get(
            'maximum_absolute_raw_difference',
            np.inf,
        )
    )
    == 0.0,
    (
        '59R2 freeze blocked: maximum raw-score '
        'renumbering difference is non-zero.'
    ),
)

require(
    float(
        invariance_verdict.get(
            'maximum_absolute_probability_difference',
            np.inf,
        )
    )
    == 0.0,
    (
        '59R2 freeze blocked: maximum probability '
        'renumbering difference is non-zero.'
    ),
)

require(
    int(
        invariance_verdict.get(
            'changed_full_cohort_decisions_total',
            -1,
        )
    )
    == 0,
    (
        '59R2 freeze blocked: full-cohort decisions '
        'changed under component renumbering.'
    ),
)

require(
    int(
        invariance_verdict.get(
            'changed_coherent_output_pixels_total',
            -1,
        )
    )
    == 0,
    (
        '59R2 freeze blocked: coherent output pixels '
        'changed under component renumbering.'
    ),
)


# ------------------------------------------------------------
# 3. Load and independently verify predictor schema
# ------------------------------------------------------------

schema = json.loads(
    SCHEMA_PATH.read_text()
)

schema_predictors = schema.get(
    'ordered_predictors'
)

require(
    schema_predictors is not None,
    (
        'predictor_schema.json lacks ordered_predictors.'
    ),
)

require(
    list(schema_predictors)
    == list(PREDICTORS),
    (
        'Frozen predictor_schema.json does not match '
        'the in-memory ordered PREDICTORS.'
    ),
)


IDENTIFIER_COLUMNS = [
    'source_group',
    'page',
    'component_key',
    'y',
    'x',
]

for identifier in IDENTIFIER_COLUMNS:

    require(
        identifier not in PREDICTORS,
        (
            '59R2 freeze blocked: identifier column '
            f'{identifier} appears in PREDICTORS.'
        ),
    )


# ------------------------------------------------------------
# 4. Hash immutable bundle objects
# ------------------------------------------------------------

model_sha256 = sha256_file(
    MODEL_PATH
)

calibrator_sha256 = sha256_file(
    CALIBRATOR_PATH
)

schema_sha256 = sha256_file(
    SCHEMA_PATH
)

replay_sha256 = sha256_file(
    REPLAY_VERDICT_PATH
)

invariance_verdict_sha256 = sha256_file(
    INVARIANCE_VERDICT_PATH
)

invariance_table_sha256 = sha256_file(
    INVARIANCE_TABLE_PATH
)

invariance_mappings_sha256 = sha256_file(
    INVARIANCE_MAPPINGS_PATH
)

invariance_sample_sha256 = sha256_file(
    INVARIANCE_SAMPLE_PATH
)


# ------------------------------------------------------------
# 5. Define one explicit runtime contract
#
# The compatibility identity is derived from this semantic
# contract rather than from a partial subset of settings.
# ------------------------------------------------------------

runtime_contract = {
    'contract_version':
        '59R2-component-invariant-runtime-v1',

    'ordered_predictors':
        list(PREDICTORS),

    'ordered_predictors_sha256':
        object_hash(
            list(PREDICTORS)
        ),

    'identifier_columns':
        IDENTIFIER_COLUMNS,

    'component_identity_role':
        'audit_join_key_only',

    'component_identity_is_predictor':
        False,

    'component_renumbering_invariance_required':
        True,

    'component_renumbering_invariance_verdict_sha256':
        invariance_verdict_sha256,

    'semantic_predictor_hash':
        invariance_verdict[
            'full_cohort_semantic_predictor_hash'
        ],

    'full_cohort_score_decision_hash':
        invariance_verdict[
            'full_cohort_score_decision_hash'
        ],

    'threshold':
        float(REMOVE_THRESHOLD),

    'connectivity':
        int(REMOVE_CONNECTIVITY),

    'minimum_run':
        int(REMOVE_MINIMUM_RUN),

    'region_mean_threshold':
        float(REMOVE_REGION_MEAN_THRESHOLD),

    'development_grouping':
        'source_group with page-disjoint assertion',

    'coherence_scope':
        'page_global',

    'coherent_output_requires_exact_coordinates':
        True,

    'promotion_authority':
        False,
}


# ------------------------------------------------------------
# 6. Compatibility identity
#
# Hash all semantics that a compatible runtime must reproduce.
# File hashes are included where the exact frozen object matters.
# ------------------------------------------------------------

compatibility_payload = {
    'runtime_contract':
        runtime_contract,

    'model_sha256':
        model_sha256,

    'calibrator_sha256':
        calibrator_sha256,

    'schema_sha256':
        schema_sha256,

    'replay_sha256':
        replay_sha256,

    'component_renumbering_invariance_verdict_sha256':
        invariance_verdict_sha256,
}

compatibility_id = (
    '59R2-COMPAT-'
    + object_hash(
        compatibility_payload
    )[:16]
)

runtime_contract[
    'compatibility_id'
] = compatibility_id


# ------------------------------------------------------------
# 7. Update policy with exact frozen identities
# ------------------------------------------------------------

policy = json.loads(
    POLICY_PATH.read_text()
)

require(
    'policy_id' in policy,
    'policy.json lacks policy_id.',
)

# If policy already carries these values, require consistency
# rather than silently overwriting contradictory semantics.
if 'threshold' in policy:

    require(
        float(policy['threshold'])
        == float(REMOVE_THRESHOLD),
        (
            'Existing policy threshold conflicts with '
            'REMOVE_THRESHOLD.'
        ),
    )

policy.update({
    'model_sha256':
        model_sha256,

    'calibrator_sha256':
        calibrator_sha256,

    'schema_sha256':
        schema_sha256,

    'replay_sha256':
        replay_sha256,

    'component_renumbering_invariance_verdict_sha256':
        invariance_verdict_sha256,

    'component_renumbering_invariance_table_sha256':
        invariance_table_sha256,

    'component_renumbering_mappings_sha256':
        invariance_mappings_sha256,

    'component_renumbering_replay_sample_sha256':
        invariance_sample_sha256,

    'component_identity_is_predictor':
        False,

    'component_identity_role':
        'audit_join_key_only',

    'component_renumbering_invariance_required':
        True,

    'component_renumbering_invariance_passed':
        True,

    'compatibility_id':
        compatibility_id,

    'promotion_authority':
        False,
})

atomic_json(
    POLICY_PATH,
    policy,
)


# policy.json changed, so record its final hash only now.
policy_sha256 = sha256_file(
    POLICY_PATH
)


# ------------------------------------------------------------
# 8. Write runtime contract
# ------------------------------------------------------------

runtime_contract.update({
    'policy_id':
        policy['policy_id'],

    'policy_sha256':
        policy_sha256,

    'model_sha256':
        model_sha256,

    'calibrator_sha256':
        calibrator_sha256,

    'schema_sha256':
        schema_sha256,

    'replay_sha256':
        replay_sha256,
})

RUNTIME_CONTRACT_PATH = (
    BUNDLE
    / 'runtime_contract.json'
)

atomic_json(
    RUNTIME_CONTRACT_PATH,
    runtime_contract,
)

runtime_contract_sha256 = sha256_file(
    RUNTIME_CONTRACT_PATH
)


# ------------------------------------------------------------
# 9. Final consistency checks before freeze manifest
# ------------------------------------------------------------

final_policy = json.loads(
    POLICY_PATH.read_text()
)

final_contract = json.loads(
    RUNTIME_CONTRACT_PATH.read_text()
)

require(
    final_policy[
        'compatibility_id'
    ]
    == final_contract[
        'compatibility_id'
    ],
    (
        'Policy/runtime compatibility_id mismatch.'
    ),
)

require(
    final_policy[
        'policy_id'
    ]
    == final_contract[
        'policy_id'
    ],
    (
        'Policy/runtime policy_id mismatch.'
    ),
)

require(
    final_policy[
        'schema_sha256'
    ]
    == sha256_file(
        SCHEMA_PATH
    ),
    'Final schema hash mismatch.',
)

require(
    final_policy[
        'model_sha256'
    ]
    == sha256_file(
        MODEL_PATH
    ),
    'Final model hash mismatch.',
)

require(
    final_policy[
        'calibrator_sha256'
    ]
    == sha256_file(
        CALIBRATOR_PATH
    ),
    'Final calibrator hash mismatch.',
)

require(
    final_policy[
        'replay_sha256'
    ]
    == sha256_file(
        REPLAY_VERDICT_PATH
    ),
    'Final replay hash mismatch.',
)

require(
    final_policy[
        'component_renumbering_invariance_verdict_sha256'
    ]
    == sha256_file(
        INVARIANCE_VERDICT_PATH
    ),
    (
        'Final component-renumbering invariance '
        'verdict hash mismatch.'
    ),
)


# ------------------------------------------------------------
# 10. Freeze manifest
#
# Include every object required to reproduce or audit the
# frozen component-invariant policy.
# ------------------------------------------------------------

freeze_files = [
    MODEL_PATH,
    CALIBRATOR_PATH,
    SCHEMA_PATH,
    POLICY_PATH,
    RUNTIME_CONTRACT_PATH,

    REPLAY_VERDICT_PATH,
    REPLAY_ROWS_PATH,
    REPLAY_PREDICTORS_PATH,
    REPLAY_EXPECTED_PATH,

    INVARIANCE_VERDICT_PATH,
    INVARIANCE_TABLE_PATH,
    INVARIANCE_MAPPINGS_PATH,
    INVARIANCE_SAMPLE_PATH,
]

for path in freeze_files:

    require(
        path.exists(),
        (
            'Freeze manifest input disappeared: '
            f'{path}'
        ),
    )


freeze_objects = []

for path in freeze_files:

    freeze_objects.append({
        'relative_path':
            str(
                path.relative_to(
                    RUN
                )
            ),

        'bytes':
            int(
                path.stat().st_size
            ),

        'sha256':
            sha256_file(
                path
            ),
    })


freeze = {
    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'bundle_version':
        '59R2-component-invariant-v2',

    'policy_id':
        final_policy[
            'policy_id'
        ],

    'compatibility_id':
        compatibility_id,

    'policy_sha256':
        policy_sha256,

    'runtime_contract_sha256':
        runtime_contract_sha256,

    'development_bundle_complete':
        True,

    'ordinary_replay_passed':
        True,

    'component_renumbering_invariance_passed':
        True,

    'component_identity_is_predictor':
        False,

    'component_identity_role':
        'audit_join_key_only',

    'promotion_authority':
        False,

    'full_cohort_semantic_predictor_hash':
        invariance_verdict[
            'full_cohort_semantic_predictor_hash'
        ],

    'full_cohort_score_decision_hash':
        invariance_verdict[
            'full_cohort_score_decision_hash'
        ],

    'renumbering_seeds':
        invariance_verdict[
            'tested_seeds'
        ],

    'renumbering_replay_rows_per_seed':
        invariance_verdict[
            'model_replay_rows_per_seed'
        ],

    'objects':
        freeze_objects,
}

FREEZE_MANIFEST_PATH = (
    BUNDLE
    / 'freeze_manifest.json'
)

atomic_json(
    FREEZE_MANIFEST_PATH,
    freeze,
)


# ------------------------------------------------------------
# 11. Verify the completed frozen bundle from disk
# ------------------------------------------------------------

freeze_reloaded = json.loads(
    FREEZE_MANIFEST_PATH.read_text()
)

for item in freeze_reloaded[
    'objects'
]:

    path = (
        RUN
        / item[
            'relative_path'
        ]
    )

    require(
        path.exists(),
        (
            'Frozen object missing after manifest write: '
            f'{path}'
        ),
    )

    require(
        int(
            path.stat().st_size
        )
        == int(
            item['bytes']
        ),
        (
            'Frozen object size changed after manifest '
            f'write: {path}'
        ),
    )

    require(
        sha256_file(
            path
        )
        == item[
            'sha256'
        ],
        (
            'Frozen object hash changed after manifest '
            f'write: {path}'
        ),
    )


# ------------------------------------------------------------
# 12. Source artefact immutability
# ------------------------------------------------------------

source_hashes_after = {
    key:
        sha256_file(
            path
        )
    for key, path in source_files.items()
}

source_hashes_before = {
    key:
        value[
            'sha256'
        ]
    for key, value in json.loads(
        (
            AUDIT
            / 'source_hashes_before.json'
        ).read_text()
    ).items()
}

require(
    source_hashes_after
    == source_hashes_before,
    (
        'A source artifact changed during 59R2.'
    ),
)

atomic_json(
    AUDIT
    / 'source_hashes_after.json',
    {
        key: {
            'path':
                str(
                    source_files[
                        key
                    ]
                ),

            'sha256':
                value,
        }
        for key, value in source_hashes_after.items()
    },
)


# ------------------------------------------------------------
# 13. Final freeze verdict
# ------------------------------------------------------------

freeze_verdict = {
    'passed':
        True,

    'bundle_version':
        freeze[
            'bundle_version'
        ],

    'policy_id':
        final_policy[
            'policy_id'
        ],

    'compatibility_id':
        compatibility_id,

    'freeze_manifest_sha256':
        sha256_file(
            FREEZE_MANIFEST_PATH
        ),

    'policyy_sha256':
        sha256_file(
            POLICY_PATH
        ),

    'runtime_contract_sha256':
        sha256_file(
            RUNTIME_CONTRACT_PATH
        ),

    'ordinary_replay_passed':
        True,

    'component_renumbering_invariance_passed':
        True,

    'component_identity_is_predictor':
        False,

    'promotion_authority':
        False,

    'source_artifacts_unchanged':
        True,

    'frozen_objects':
        len(
            freeze_objects
        ),
}

atomic_json(
    BUNDLE
    / 'freeze_verdict.json',
    freeze_verdict,
)

print(
    json.dumps(
        freeze_verdict,
        indent=2,
    )
)

print(
    '\n59R2 component-invariant development bundle: FROZEN'
)

print(
    'Policy ID:',
    final_policy['policy_id'],
)

print(
    'Compatibility ID:',
    compatibility_id,
)

print(
    'Frozen objects:',
    len(freeze_objects),
)

print(
    'Promotion authority:',
    False,
)

## 14. Challenger verdict and concise comparison charts

In [ ]:
# ============================================================
# Cell 14 - Final 59R2 challenger verdict and comparison summary
#
# This cell:
#   1. verifies the completed v2 frozen development bundle;
#   2. verifies replay and component-renumbering invariance;
#   3. calculates the 59R1-versus-59R2 comparison from the
#      memory-bounded Cell 11 outputs;
#   4. applies page-global coherence one page at a time;
#   5. writes page and aggregate comparison tables;
#   6. writes the final challenger verdict and executive summary;
#   7. does not promote 59R2 automatically.
# ============================================================

import gc
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 0. Required completed artefacts
# ------------------------------------------------------------

FREEZE_MANIFEST_PATH = (
    BUNDLE
    / 'freeze_manifest.json'
)

FREEZE_VERDICT_PATH = (
    BUNDLE
    / 'freeze_verdict.json'
)

RUNTIME_CONTRACT_PATH = (
    BUNDLE
    / 'runtime_contract.json'
)

POLICY_PATH = (
    BUNDLE
    / 'policy.json'
)

INVARIANCE_VERDICT_PATH = (
    AUDIT
    / 'component_renumbering_invariance_verdict.json'
)

REPLAY_VERDICT_PATH = (
    REPLAY
    / 'replay_test.json'
)


required_cell14_files = [
    FREEZE_MANIFEST_PATH,
    FREEZE_VERDICT_PATH,
    RUNTIME_CONTRACT_PATH,
    POLICY_PATH,
    INVARIANCE_VERDICT_PATH,
    REPLAY_VERDICT_PATH,
]

missing_cell14_files = [
    str(path)
    for path in required_cell14_files
    if not path.is_file()
]

require(
    not missing_cell14_files,
    (
        'Cell 14 cannot create a challenger verdict because '
        f'required artefacts are missing: {missing_cell14_files}'
    ),
)


# ------------------------------------------------------------
# 1. Load final frozen identities
# ------------------------------------------------------------

freeze_manifest = json.loads(
    FREEZE_MANIFEST_PATH.read_text()
)

freeze_verdict = json.loads(
    FREEZE_VERDICT_PATH.read_text()
)

runtime_contract = json.loads(
    RUNTIME_CONTRACT_PATH.read_text()
)

final_policy = json.loads(
    POLICY_PATH.read_text()
)

invariance_verdict = json.loads(
    INVARIANCE_VERDICT_PATH.read_text()
)

replay_verdict = json.loads(
    REPLAY_VERDICT_PATH.read_text()
)


require(
    freeze_verdict.get(
        'passed'
    ) is True,
    (
        'Cell 14 blocked: Cell 13 freeze verdict '
        'did not PASS.'
    ),
)

require(
    replay_verdict.get(
        'passed'
    ) is True,
    (
        'Cell 14 blocked: ordinary replay did not PASS.'
    ),
)

require(
    invariance_verdict.get(
        'passed'
    ) is True,
    (
        'Cell 14 blocked: component-renumbering '
        'invariance did not PASS.'
    ),
)

require(
    invariance_verdict.get(
        'changed_full_cohort_decisions_total'
    )
    == 0,
    (
        'Cell 14 blocked: component renumbering '
        'changed one or more full-cohort decisions.'
    ),
)

require(
    invariance_verdict.get(
        'changed_coherent_output_pixels_total'
    )
    == 0,
    (
        'Cell 14 blocked: component renumbering '
        'changed the coherent output.'
    ),
)

require(
    final_policy.get(
        'component_identity_is_predictor'
    ) is False,
    (
        'Cell 14 blocked: final policy still records '
        'component identity as a predictor.'
    ),
)

require(
    final_policy.get(
        'promotion_authority'
    ) is False,
    (
        'Cell 14 blocked: unexpected promotion authority.'
    ),
)

require(
    final_policy[
        'policy_id'
    ]
    == runtime_contract[
        'policy_id'
    ],
    (
        'Policy/runtime policy_id mismatch.'
    ),
)

require(
    final_policy[
        'compatibility_id'
    ]
    == runtime_contract[
        'compatibility_id'
    ],
    (
        'Policy/runtime compatibility_id mismatch.'
    ),
)


# ------------------------------------------------------------
# 2. Required in-memory Cell 11 outputs
# ------------------------------------------------------------

required_runtime_objects = [
    'cohort',
    'prob2',
    'raw2',
    'p_remove_59r1',
    'raw_59r1',
    'safe_remove_59r1',
    'THRESHOLD_59R1',
    'THRESHOLD_59R2',
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

require(
    not missing_runtime_objects,
    (
        'Cell 14 requires completed Cell 11 runtime objects. '
        f'Missing: {missing_runtime_objects}'
    ),
)


# ------------------------------------------------------------
# 3. Frozen coherence helper
#
# Apply separately to each page. This avoids a second full
# 12.7-million-row DataFrame.
# ------------------------------------------------------------

def coherent_safe_remove_from_page(
    y,
    x,
    probability,
    threshold,
    height,
    width,
    connectivity,
    minimum_run,
    region_mean_threshold,
):
    """
    Apply the frozen page-global coherent SAFE_REMOVE rule.

    Returns only the decision at the supplied in-mask rows.
    """

    y = np.asarray(
        y,
        dtype=np.int32,
    )

    x = np.asarray(
        x,
        dtype=np.int32,
    )

    probability = np.asarray(
        probability,
        dtype=np.float32,
    )

    require(
        len(y)
        == len(x)
        == len(probability),
        'Page coherence input length mismatch.',
    )

    require(
        np.isfinite(
            probability
        ).all(),
        (
            'Non-finite p_remove supplied to '
            'page-global coherence.'
        ),
    )

    provisional = (
        probability
        >= float(threshold)
    )

    candidate = np.zeros(
        (
            int(height),
            int(width),
        ),
        dtype=np.uint8,
    )

    candidate[
        y[provisional],
        x[provisional],
    ] = 1

    (
        region_count,
        labels,
        stats,
        _,
    ) = cv2.connectedComponentsWithStats(
        candidate,
        int(connectivity),
    )

    probability_map = np.full(
        (
            int(height),
            int(width),
        ),
        np.nan,
        dtype=np.float32,
    )

    probability_map[
        y,
        x,
    ] = probability

    safe = np.zeros(
        (
            int(height),
            int(width),
        ),
        dtype=bool,
    )

    for region_id in range(
        1,
        region_count,
    ):

        region = (
            labels
            == region_id
        )

        size = int(
            stats[
                region_id,
                cv2.CC_STAT_AREA,
            ]
        )

        values = probability_map[
            region
        ]

        finite = values[
            np.isfinite(
                values
            )
        ]

        passes = bool(
            size
            >= int(
                minimum_run
            )

            and len(
                finite
            )
            == size

            and float(
                finite.mean()
            )
            >= float(
                region_mean_threshold
            )

            and float(
                finite.min()
            )
            >= float(
                threshold
            )
        )

        if passes:
            safe[
                region
            ] = True

    return safe[
        y,
        x,
    ]


# ------------------------------------------------------------
# 4. 59R2 coherence constants
# ------------------------------------------------------------

COHERENCE_59R2_CONNECTIVITY = int(
    runtime_contract[
        'connectivity'
    ]
)

COHERENCE_59R2_MINIMUM_RUN = int(
    runtime_contract[
        'minimum_run'
    ]
)

COHERENCE_59R2_REGION_MEAN = float(
    runtime_contract[
        'region_mean_threshold'
    ]
)


# ------------------------------------------------------------
# 5. 59R1 coherence constants
#
# Use the frozen incumbent policy, not the 59R2 settings by
# assumption.
# ------------------------------------------------------------

policy_59r1 = json.loads(
    (
        R59
        / 'bundle'
        / 'policy.json'
    ).read_text()
)

COHERENCE_59R1_CONNECTIVITY = int(
    policy_59r1.get(
        'connectivity',
        8,
    )
)

COHERENCE_59R1_MINIMUM_RUN = int(
    policy_59r1.get(
        'minimum_run',
        policy_59r1.get(
            'minimum_region_size',
            2,
        ),
    )
)

COHERENCE_59R1_REGION_MEAN = float(
    policy_59r1.get(
        'region_mean_threshold',
        THRESHOLD_59R1,
    )
)


# ------------------------------------------------------------
# 6. Page roster and local page dimensions
#
# Use actual 60R2 observed page shapes.
# ------------------------------------------------------------

page_categories = (
    cohort[
        'page'
    ].cat.categories
    if hasattr(
        cohort[
            'page'
        ],
        'cat',
    )
    else sorted(
        cohort[
            'page'
        ].astype(
            str
        ).unique()
    )
)

page_dimension_lookup = {}

for page_value in page_categories:

    page = str(
        page_value
    )

    observed_path = (
        R60
        / 'pages'
        / page
        / 'observed.png'
    )

    require(
        observed_path.is_file(),
        (
            'Cell 14 cannot resolve page dimensions: '
            f'{observed_path}'
        ),
    )

    observed = cv2.imread(
        str(
            observed_path
        ),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        observed is not None,
        (
            'Unreadable 60R2 observed page: '
            f'{observed_path}'
        ),
    )

    page_dimension_lookup[
        page
    ] = (
        int(
            observed.shape[
                0
            ]
        ),
        int(
            observed.shape[
                1
            ]
        ),
    )

    del observed


# ------------------------------------------------------------
# 7. Page-by-page incumbent/challenger comparison
#
# Only one page's coherence arrays are materialised at a time.
# ------------------------------------------------------------

page_rows = []

total_safe_59r1 = 0
total_safe_59r2 = 0
total_changed = 0

page_values = cohort[
    'page'
].astype(
    str
).to_numpy()

y_all = cohort[
    'y'
].to_numpy(
    dtype=np.int32,
    copy=False,
)

x_all = cohort[
    'x'
].to_numpy(
    dtype=np.int32,
    copy=False,
)

p59r1_all = np.asarray(
    p_remove_59r1,
    dtype=np.float64,
)

p59r2_all = np.asarray(
    prob2,
    dtype=np.float32,
)

stored_safe_59r1 = np.asarray(
    safe_remove_59r1,
    dtype=bool,
)


for page in sorted(
    set(
        page_values
    )
):

    row_positions = np.flatnonzero(
        page_values
        == page
    )

    require(
        len(
            row_positions
        )
        > 0,
        (
            f'{page}: empty cohort page.'
        ),
    )

    height, width = (
        page_dimension_lookup[
            page
        ]
    )

    y_page = y_all[
        row_positions
    ]

    x_page = x_all[
        row_positions
    ]

    p59r1_page = p59r1_all[
        row_positions
    ]

    p59r2_page = p59r2_all[
        row_positions
    ]

    # Reconstruct incumbent coherence as an integrity check.
    safe59r1_replayed = (
        coherent_safe_remove_from_page(
            y=y_page,
            x=x_page,
            probability=p59r1_page,
            threshold=THRESHOLD_59R1,
            height=height,
            width=width,
            connectivity=COHERENCE_59R1_CONNECTIVITY,
            minimum_run=COHERENCE_59R1_MINIMUM_RUN,
            region_mean_threshold=COHERENCE_59R1_REGION_MEAN,
        )
    )

    safe59r1_stored_page = (
        stored_safe_59r1[
            row_positions
        ]
    )

    require(
        np.array_equal(
            safe59r1_replayed,
            safe59r1_stored_page,
        ),
        (
            f'{page}: frozen 59R1 coherence replay '
            'does not reproduce the stored 60R2 '
            'SAFE_REMOVE decisions.'
        ),
    )

    safe59r2_page = (
        coherent_safe_remove_from_page(
            y=y_page,
            x=x_page,
            probability=p59r2_page,
            threshold=THRESHOLD_59R2,
            height=height,
            width=width,
            connectivity=COHERENCE_59R2_CONNECTIVITY,
            minimum_run=COHERENCE_59R2_MINIMUM_RUN,
            region_mean_threshold=COHERENCE_59R2_REGION_MEAN,
        )
    )

    safe_59r1_count = int(
        safe59r1_stored_page.sum()
    )

    safe_59r2_count = int(
        safe59r2_page.sum()
    )

    changed_count = int(
        np.sum(
            safe59r1_stored_page
            != safe59r2_page
        )
    )

    became_safe_remove = int(
        np.sum(
            (
                ~safe59r1_stored_page
            )
            & safe59r2_page
        )
    )

    ceased_safe_remove = int(
        np.sum(
            safe59r1_stored_page
            & (
                ~safe59r2_page
            )
        )
    )

    total_safe_59r1 += (
        safe_59r1_count
    )

    total_safe_59r2 += (
        safe_59r2_count
    )

    total_changed += (
        changed_count
    )

    page_rows.append({
        'page':
            page,

        'pixels':
            int(
                len(
                    row_positions
                )
            ),

        'safe_remove_59r1':
            safe_59r1_count,

        'safe_remove_59r2':
            safe_59r2_count,

        'delta':
            int(
                safe_59r2_count
                - safe_59r1_count
            ),

        'changed_decisions':
            changed_count,

        'became_safe_remove':
            became_safe_remove,

        'ceased_safe_remove':
            ceased_safe_remove,

        'mean_p_remove_59r1':
            float(
                np.mean(
                    p59r1_page
                )
            ),

        'mean_p_remove_59r2':
            float(
                np.mean(
                    p59r2_page
                )
            ),
    })

    del row_positions
    del y_page
    del x_page
    del p59r1_page
    del p59r2_page
    del safe59r1_replayed
    del safe59r1_stored_page
    del safe59r2_page

    gc.collect()


# ------------------------------------------------------------
# 8. Comparison outputs
# ------------------------------------------------------------

page_compare = pd.DataFrame(
    page_rows
).sort_values(
    'page'
).reset_index(
    drop=True
)

require(
    int(
        page_compare[
            'pixels'
        ].sum()
    )
    == len(cohort),
    (
        'Page comparison does not cover the complete cohort.'
    ),
)

require(
    int(
        page_compare[
            'safe_remove_59r1'
        ].sum()
    )
    == int(
        stored_safe_59r1.sum()
    ),
    (
        '59R1 page-level aggregate does not reproduce '
        'the stored complete-cohort SAFE_REMOVE count.'
    ),
)

comparison_summary = {
    'pages':
        int(
            len(
                page_compare
            )
        ),

    'pixels':
        int(
            len(
                cohort
            )
        ),

    '59r1_safe_remove_pixels':
        int(
            total_safe_59r1
        ),

    '59r2_safe_remove_pixels':
        int(
            total_safe_59r2
        ),

    'safe_remove_delta':
        int(
            total_safe_59r2
            - total_safe_59r1
        ),

    'changed_pixel_decisions':
        int(
            total_changed
        ),

    'became_safe_remove':
        int(
            page_compare[
                'became_safe_remove'
            ].sum()
        ),

    'ceased_safe_remove':
        int(
            page_compare[
                'ceased_safe_remove'
            ].sum()
        ),

    'pages_with_any_change':
        int(
            (
                page_compare[
                    'changed_decisions'
                ]
                > 0
            ).sum()
        ),

    'pages_with_more_safe_remove_59r2':
        int(
            (
                page_compare[
                    'delta'
                ]
                > 0
            ).sum()
        ),

    'pages_with_less_safe_remove_59r2':
        int(
            (
                page_compare[
                    'delta'
                ]
                < 0
            ).sum()
        ),

    'pages_with_equal_safe_remove':
        int(
            (
                page_compare[
                    'delta'
                ]
                == 0
            ).sum()
        ),
}

page_compare.to_csv(
    COMPARISON
    / '59r1_vs_59r2_by_page.csv',
    index=False,
)

atomic_json(
    COMPARISON
    / 'comparison_summary.json',
    comparison_summary,
)


# ------------------------------------------------------------
# 9. Cohort-count assertions
# ------------------------------------------------------------

if (
    'EXPECTED_REVIEWED_PAGES'
    in globals()
):

    require(
        int(
            comparison_summary[
                'pages'
            ]
        )
        == int(
            EXPECTED_REVIEWED_PAGES
        ),
        (
            'Complete reviewed cohort page count mismatch: '
            f'{comparison_summary["pages"]} versus '
            f'{EXPECTED_REVIEWED_PAGES}.'
        ),
    )


# ------------------------------------------------------------
# 10. Final challenger verdict
#
# This is intentionally CHALLENGER_BUILT, not PROMOTED.
# ------------------------------------------------------------

verdict = {
    'verdict':
        'CHALLENGER_BUILT',

    'promotion_authority':
        False,

    'policy_id':
        final_policy[
            'policy_id'
        ],

    'compatibility_id':
        final_policy[
            'compatibility_id'
        ],

    'bundle_version':
        freeze_manifest[
            'bundle_version'
        ],

    'freeze_verdict_passed':
        True,

    'freeze_manifest_sha256':
        sha256_file(
            FREEZE_MANIFEST_PATH
        ),

    'runtime_contract_sha256':
        sha256_file(
            RUNTIME_CONTRACT_PATH
        ),

    'policy_sha256':
        sha256_file(
            POLICY_PATH
        ),

    'component_id_removed_from_predictors':
        True,

    'component_key_removed_from_predictors':
        True,

    'component_identity_audit_join_key_only':
        True,

    'page_source_group_separated_development':
        True,

    'model_refit':
        True,

    'isotonic_calibrator_refit_from_oof_scores':
        True,

    'exact_replay_passed':
        bool(
            replay_verdict[
                'passed'
            ]
        ),

    'renumbering_invariance_passed':
        bool(
            invariance_verdict[
                'passed'
            ]
        ),

    'renumbering_changed_full_cohort_decisions':
        int(
            invariance_verdict[
                'changed_full_cohort_decisions_total'
            ]
        ),

    'renumbering_changed_coherent_output_pixels':
        int(
            invariance_verdict[
                'changed_coherent_output_pixels_total'
            ]
        ),

    'complete_reviewed_cohort_pages':
        int(
            comparison_summary[
                'pages'
            ]
        ),

    'complete_reviewed_cohort_pixels':
        int(
            comparison_summary[
                'pixels'
            ]
        ),

    '59r1_safe_remove_pixels':
        int(
            comparison_summary[
                '59r1_safe_remove_pixels'
            ]
        ),

    '59r2_safe_remove_pixels':
        int(
            comparison_summary[
                '59r2_safe_remove_pixels'
            ]
        ),

    'safe_remove_delta':
        int(
            comparison_summary[
                'safe_remove_delta'
            ]
        ),

    'changed_pixel_decisions':
        int(
            comparison_summary[
                'changed_pixel_decisions'
            ]
        ),

    'became_safe_remove':
        int(
            comparison_summary[
                'became_safe_remove'
            ]
        ),

    'ceased_safe_remove':
        int(
            comparison_summary[
                'ceased_safe_remove'
            ]
        ),

    'pages_with_any_change':
        int(
            comparison_summary[
                'pages_with_any_change'
            ]
        ),

    'source_artifacts_unchanged':
        True,

    'next_gate':
        (
            'Independent safety/benefit review, '
            'source-group stability review and documented '
            'promotion decision. No automatic promotion.'
        ),
}

atomic_json(
    RUN
    / '59r2_verdict.json',
    verdict,
)


# ------------------------------------------------------------
# 11. Page-level comparison chart
# ------------------------------------------------------------

q = page_compare.sort_values(
    'delta'
)

plt.figure(
    figsize=(
        11,
        max(
            6,
            0.055
            * len(q),
        ),
    )
)

plt.barh(
    q[
        'page'
    ],
    q[
        'delta'
    ],
)

plt.axvline(
    0,
    color='black',
    linewidth=1,
)

plt.xlabel(
    '59R2 minus 59R1 coherent SAFE_REMOVE pixels'
)

plt.title(
    '59R2 component-invariant challenger versus frozen 59R1'
)

plt.tight_layout()

plt.savefig(
    CHARTS
    / '59r2_vs_59r1_safe_remove_delta_by_page.png',
    dpi=160,
)

plt.close()


# ------------------------------------------------------------
# 12. Compact aggregate comparison chart
# ------------------------------------------------------------

plt.figure(
    figsize=(
        7,
        5,
    )
)

plt.bar(
    [
        '59R1',
        '59R2',
    ],
    [
        comparison_summary[
            '59r1_safe_remove_pixels'
        ],
        comparison_summary[
            '59r2_safe_remove_pixels'
        ],
    ],
)

plt.ylabel(
    'Coherent SAFE_REMOVE pixels'
)

plt.title(
    'Complete reviewed cohort'
)

plt.tight_layout()

plt.savefig(
    CHARTS
    / '59r1_vs_59r2_total_safe_remove.png',
    dpi=170,
)

plt.close()


# ------------------------------------------------------------
# 13. Executive summary
# ------------------------------------------------------------

lines = [
    '59R2 COMPONENT-INVARIANT REMOVE CHALLENGER',
    '',

    f'Policy ID: '
    f'{final_policy["policy_id"]}',

    f'Compatibility ID: '
    f'{final_policy["compatibility_id"]}',

    f'Bundle version: '
    f'{freeze_manifest["bundle_version"]}',

    f'Model family: {BEST}',

    f'Development rows: '
    f'{len(dev):,}',

    f'Development pages: '
    f'{dev.page.nunique()}',

    f'Development source groups: '
    f'{dev.source_group.nunique()}',

    f'Ordered predictors: '
    f'{len(PREDICTORS)}',

    'component_id predictor: False',

    'component_key predictor: False',

    'Component identity role: audit/join only',

    f'Exact replay: '
    f'{replay_verdict["passed"]}',

    f'Renumbering invariance: '
    f'{invariance_verdict["passed"]}',

    f'Renumbering full-cohort changed decisions: '
    f'{invariance_verdict["changed_full_cohort_decisions_total"]}',

    f'Renumbering changed coherent-output pixels: '
    f'{invariance_verdict["changed_coherent_output_pixels_total"]}',

    '',

    f'Reviewed cohort pages: '
    f'{comparison_summary["pages"]}',

    f'Reviewed cohort pixels: '
    f'{comparison_summary["pixels"]:,}',

    f'59R1 SAFE_REMOVE pixels: '
    f'{comparison_summary["59r1_safe_remove_pixels"]:,}',

    f'59R2 SAFE_REMOVE pixels: '
    f'{comparison_summary["59r2_safe_remove_pixels"]:,}',

    f'59R2 - 59R1 SAFE_REMOVE delta: '
    f'{comparison_summary["safe_remove_delta"]:,}',

    f'Changed pixel decisions: '
    f'{comparison_summary["changed_pixel_decisions"]:,}',

    f'Pixels becoming SAFE_REMOVE: '
    f'{comparison_summary["became_safe_remove"]:,}',

    f'Pixels ceasing SAFE_REMOVE: '
    f'{comparison_summary["ceased_safe_remove"]:,}',

    f'Pages with any incumbent/challenger difference: '
    f'{comparison_summary["pages_with_any_change"]}',

    '',

    'STATUS: CHALLENGER BUILT.',

    (
        '59R2 is a separately versioned '
        'component-invariant challenger.'
    ),

    (
        '59R2 is not a patch to 59R1 and does not '
        'overwrite the incumbent.'
    ),

    (
        'The permanent component-renumbering test '
        'requires exactly zero predictor, score, '
        'probability, threshold-decision and coherent-output '
        'change.change.'
    ),

    (
        'No automatic production or promotion authority '
        'is conferred by this result.'
    ),

    (
        'Next gate: independent safety/benefit review, '
        'source-group stability assessment and a documented '
        'promotion decision.'
    ),
]

(
    RUN
    / 'executive_summary.txt'
).write_text(
    chr(10).join(
        lines
    )
    + chr(10)
)

print(
    chr(10).join(
        lines
    )
)


# ------------------------------------------------------------
# 14. Final console verdict
# ------------------------------------------------------------

print(
    '\nFINAL 59R2 VERDICT:'
)

print(
    json.dumps(
        verdict,
        indent=2,
    )
)

## 15. Standard Final Upload Package

In [ ]:
# ============================================================
# Cell 15 - Standard Final Upload Package
#
# Purpose:
#   Create a compact, self-contained review package for the
#   completed 59R2 component-invariant challenger.
#
# Include:
#   - executive and decision records;
#   - development evidence;
#   - final frozen-v2 bundle metadata;
#   - ordinary replay evidence;
#   - permanent component-renumbering evidence;
#   - source/provenance evidence;
#   - complete 59R1-versus-59R2 cohort comparison;
#   - concise review charts.
#
# Exclude:
#   - 12.7M-row pixel tables;
#   - local memmaps;
#   - copied development datasets;
#   - heavyweight temporary arrays;
#   - model/calibrator binary duplication.
#
# The durable model and calibrator remain in bundle/ and are
# content-addressed by freeze_manifest.json.
#
# promotion_authority remains False.
# ============================================================

import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# 0. Required final verdicts
# ------------------------------------------------------------

CHALLENGER_VERDICT_PATH = (
    RUN
    / '59r2_verdict.json'
)

FREEZE_VERDICT_PATH = (
    BUNDLE
    / 'freeze_verdict.json'
)

FREEZE_MANIFEST_PATH = (
    BUNDLE
    / 'freeze_manifest.json'
)

POLICY_PATH = (
    BUNDLE
    / 'policy.json'
)

SCHEMA_PATH = (
    BUNDLE
    / 'predictor_schema.json'
)

RUNTIME_CONTRACT_PATH = (
    BUNDLE
    / 'runtime_contract.json'
)

REPLAY_VERDICT_PATH = (
    REPLAY
    / 'replay_test.json'
)

INVARIANCE_VERDICT_PATH = (
    AUDIT
    / 'component_renumbering_invariance_verdict.json'
)


required_verdict_files = [
    CHALLENGER_VERDICT_PATH,
    FREEZE_VERDICT_PATH,
    FREEZE_MANIFEST_PATH,
    POLICY_PATH,
    SCHEMA_PATH,
    RUNTIME_CONTRACT_PATH,
    REPLAY_VERDICT_PATH,
    INVARIANCE_VERDICT_PATH,
]

missing_verdict_files = [
    str(path)
    for path in required_verdict_files
    if not path.is_file()
]

require(
    not missing_verdict_files,
    (
        'Final Upload Package blocked because required '
        f'59R2 records are missing: {missing_verdict_files}'
    ),
)


# ------------------------------------------------------------
# 1. Load and verify final status before packaging
# ------------------------------------------------------------

challenger_verdict = json.loads(
    CHALLENGER_VERDICT_PATH.read_text()
)

freeze_verdict = json.loads(
    FREEZE_VERDICT_PATH.read_text()
)

freeze_manifest = json.loads(
    FREEZE_MANIFEST_PATH.read_text()
)

policy = json.loads(
    POLICY_PATH.read_text()
)

runtime_contract = json.loads(
    RUNTIME_CONTRACT_PATH.read_text()
)

replay_verdict = json.loads(
    REPLAY_VERDICT_PATH.read_text()
)

invariance_verdict = json.loads(
    INVARIANCE_VERDICT_PATH.read_text()
)


require(
    challenger_verdict.get(
        'verdict'
    )
    == 'CHALLENGER_BUILT',
    (
        'Final Upload Package blocked: final 59R2 '
        'challenger verdict is not CHALLENGER_BUILT.'
    ),
)

require(
    freeze_verdict.get(
        'passed'
    ) is True,
    (
        'Final Upload Package blocked: freeze verdict '
        'did not PASS.'
    ),
)

require(
    replay_verdict.get(
        'passed'
    ) is True,
    (
        'Final Upload Package blocked: ordinary replay '
        'did not PASS.'
    ),
)

require(
    invariance_verdict.get(
        'passed'
    ) is True,
    (
        'Final Upload Package blocked: permanent '
        'component-renumbering invariance did not PASS.'
    ),
)

require(
    invariance_verdict.get(
        'changed_full_cohort_decisions_total'
    )
    == 0,
    (
        'Final Upload Package blocked: component '
        'renumbering changed full-cohort decisions.'
    ),
)

require(
    invariance_verdict.get(
        'changed_coherent_output_pixels_total'
    )
    == 0,
    (
        'Final Upload Package blocked: component '
        'renumbering changed coherent output pixels.'
    ),
)

require(
    policy.get(
        'component_identity_is_predictor'
    ) is False,
    (
        'Final Upload Package blocked: final policy '
        'does not explicitly exclude component identity.'
    ),
)

require(
    policy.get(
        'promotion_authority'
    ) is False,
    (
        'Unexpected promotion authority in policy.'
    ),
)

require(
    challenger_verdict.get(
        'promotion_authority'
    ) is False,
    (
        'Unexpected promotion authority in final '
        'challenger verdict.'
    ),
)

require(
    policy[
        'policy_id'
    ]
    == runtime_contract[
        'policy_id'
    ]
    == challenger_verdict[
        'policy_id'
    ],
    (
        'policy_id differs across final 59R2 records.'
    ),
)

require(
    policy[
        'compatibility_id'
    ]
    == runtime_contract[
        'compatibility_id'
    ]
    == challenger_verdict[
        'compatibility_id'
    ],
    (
        'compatibility_id differs across final '
        '59R2 records.'
    ),
)


# ------------------------------------------------------------
# 2. Clean the upload staging directory
#
# UPLOAD belongs only to this run, but remove any files left
# by a failed/repeated attempt at Cell 15.
# ------------------------------------------------------------

if UPLOAD.exists():

    for path in sorted(
        UPLOAD.rglob('*'),
        reverse=True,
    ):

        if path.is_file():
            path.unlink()

        elif path.is_dir():

            try:
                path.rmdir()

            except OSError:
                pass

UPLOAD.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 3. Define REQUIRED curated review artefacts
#
# These must all exist. Missing files invalidate packaging.
# ------------------------------------------------------------

REQUIRED_UPLOAD_ARTEFACTS = [
    # --------------------------------------------------------
    # Executive / final decision
    # --------------------------------------------------------

    RUN
    / 'executive_summary.txt',

    RUN
    / '59r2_verdict.json',

    # --------------------------------------------------------
    # Development evidence
    # --------------------------------------------------------

    RUN
    / 'development_metrics.json',

    TABLES
    / 'model_comparison.csv',

    TABLES
    / 'fold_results.csv',

    # --------------------------------------------------------
    # Frozen v2 challenger identity
    # --------------------------------------------------------

    BUNDLE
    / 'policy.json',

    BUNDLE
    / 'predictor_schema.json',

    BUNDLE
    / 'runtime_contract.json',

    BUNDLE
    / 'freeze_manifest.json',

    BUNDLE
    / 'freeze_verdict.json',

    # --------------------------------------------------------
    # Ordinary replay evidence
    # --------------------------------------------------------

    REPLAY
    / 'replay_test.json',

    # --------------------------------------------------------
    # Permanent component-renumbering proof
    # --------------------------------------------------------

    AUDIT
    / 'component_renumbering_invariance.csv',

    AUDIT
    / 'component_renumbering_invariance_verdict.json',

    AUDIT
    / 'component_renumbering_mappings.csv',

    AUDIT
    / 'component_renumbering_replay_sample.csv',

    # --------------------------------------------------------
    # Source immutability / provenance
    # --------------------------------------------------------

    AUDIT
    / 'source_hashes_before.json',

    AUDIT
    / 'source_hashes_after.json',

    # --------------------------------------------------------
    # Cell 11 complete-cohort computational record
    # --------------------------------------------------------

    AUDIT
    / 'complete_cohort_matrix_and_memory.json',

    # --------------------------------------------------------
    # Complete frozen-incumbent versus challenger comparison
    # --------------------------------------------------------

    COMPARISON
    / 'comparison_summary.json',

    COMPARISON
    / '59r1_vs_59r2_by_page.csv',

    # --------------------------------------------------------
    # Review charts
    # --------------------------------------------------------

    CHARTS
    / '59r2_vs_59r1_safe_remove_delta_by_page.png',

    CHARTS
    / '59r1_vs_59r2_total_safe_remove.png',
]


# ------------------------------------------------------------
# 4. Define OPTIONAL audit records
#
# These are valuable when present, but some depend on which
# corrected version of Cells 3-4 was executed.
# ------------------------------------------------------------

OPTIONAL_UPLOAD_ARTEFACTS = [
    AUDIT
    / 'source_lineage.json',

    AUDIT
    / 'target_construction.json',

    AUDIT
    / 'historical_label_counts.csv',

    AUDIT
    / 'remove_target_counts.csv',

    TABLES
    / 'complete_cohort_component_features.parquet',
]


# ------------------------------------------------------------
# 5. Ensure every REQUIRED review artifact exists
# ------------------------------------------------------------

missing_required = [
    str(path)
    for path in REQUIRED_UPLOAD_ARTEFACTS
    if not path.is_file()
]

require(
    not missing_required,
    (
        'Final Upload Package blocked because required '
        f'review artefacts are missing: {missing_required}'
    ),
)


# ------------------------------------------------------------
# 6. Package files preserving run-relative directory structure
#
# Do not flatten names. Keeping bundle/, audit/, comparison/,
# tables/ and charts/ makes the review package interpretable.
# ------------------------------------------------------------

manifest_rows = []


def package_file(
    src,
    required=True,
    category=None,
):

    src = Path(
        src
    )

    if not src.is_file():

        if required:

            raise RuntimeError(
                'Required upload artifact missing: '
                f'{src}'
            )

        return False

    require(
        RUN
        in src.parents,
        (
            'Upload artifact lies outside the current '
            f'59R2 run: {src}'
        ),
    )

    relative_path = (
        src.relative_to(
            RUN
        )
    )

    destination = (
        UPLOAD
        / relative_path
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        destination,
    )

    source_sha256 = (
        sha256_file(
            src
        )
    )

    copied_sha256 = (
        sha256_file(
            destination
        )
    )

    require(
        source_sha256
        == copied_sha256,
        (
            'Upload copy hash mismatch: '
            f'{relative_path}'
        ),
    )

    require(
        int(
            src.stat().st_size
        )
        == int(
            destination.stat().st_size
        ),
        (
            'Upload copy size mismatch: '
            f'{relative_path}'
        ),
    )

    manifest_rows.append({
        'relative_path':
            str(
                relative_path
            ),

        'category':
            category,

        'required':
            bool(
                required
            ),

        'bytes':
            int(
                destination.stat().st_size
            ),

        'sha256':
            copied_sha256,

        'source_path':
            str(
                src
            ),
    })

    return True


# ------------------------------------------------------------
# 7. Package required files
# ------------------------------------------------------------

for src in REQUIRED_UPLOAD_ARTEFACTS:

    relative = str(
        src.relative_to(
            RUN
        )
    )

    if relative.startswith(
        'bundle/'
    ):

        category = (
            'frozen_bundle_identity'
        )

    elif relative.startswith(
        'replay/'
    ):

        category = (
            'ordinary_replay'
        )

    elif relative.startswith(
        'audit/'
    ):

        category = (
            'audit_and_invariance'
        )

    elif relative.startswith(
        'comparison_59R1/'
    ):

        category = (
            'incumbent_challenger_comparison'
        )

    elif relative.startswith(
        'tables/'
    ):

        category = (
            'development_evidence'
        )

    elif relative.startswith(
        'charts/'
    ):

        category = (
            'review_visual'
        )

    else:

        category = (
            'executive_decision'
        )

    package_file(
        src,
        required=True,
        category=category,
    )


# ------------------------------------------------------------
# 8. Package available optional records
# ------------------------------------------------------------

optional_packaged = []

optional_missing = []

for src in OPTIONAL_UPLOAD_ARTEFACTS:

    if src.is_file():

        package_file(
            src,
            required=False,
            category='supplementary_audit',
        )

        optional_packaged.append(
            str(
                src.relative_to(
                    RUN
                )
            )
        )

    else:

        optional_missing.append(
            str(
                src.relative_to(
                    RUN
                )
            )
        )


# ------------------------------------------------------------
# 9. Explicitly record heavyweight objects NOT copied
#
# The binaries remain in the durable run. The freeze manifest
# hashes them. The 12.7M-row prediction populations and local
# memmaps likewise remain outside the review ZIP.
# ------------------------------------------------------------

durable_not_copied = []

for path in [
    BUNDLE
    / 'remove_model.joblib',

    BUNDLE
    / 'remove_calibrator.joblib',

    TABLES
    / '59r2_development_dataset.parquet',

    TABLES
    / '59r2_oof_predictions.parquet',

    COMPARISON
    / '59r1_vs_59r2_pixel_decisions.parquet',
\]:

    if path.is_file():

        durable_not_copied.append({
            'relative_path':
                str(
                    path.relative_to(
                        RUN
                    )
                ),

            'bytes':
                int(
                    path.stat().st_size
                ),

            'sha256':
                sha256_file(
                    path
                ),

            'reason':
                (
                    'Durable heavyweight/working artefact; '
                    'not duplicated in review package.'
                ),
        })


# ------------------------------------------------------------
# 10. Upload manifest
# ------------------------------------------------------------

upload_manifest = pd.DataFrame(
    manifest_rows
).sort_values(
    'relative_path'
)

require(
    len(
        upload_manifest
    )
    == len(
        REQUIRED_UPLOAD_ARTEFACTS
    )
    + len(
        optional_packaged
    ),
    (
        'Upload manifest row count does not match '
        'packaged review artefacts.'
    ),
)

UPLOAD_MANIFEST_PATH = (
    UPLOAD
    / 'upload_manifest.csv'
)

upload_manifest.to_csv(
    UPLOAD_MANIFEST_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Durable-object inventory
# ------------------------------------------------------------

DURABLE_INVENTORY_PATH = (
    UPLOAD
    / 'durable_not_copied.csv'
)

pd.DataFrame(
    durable_not_copied,
    columns=[
        'relative_path',
        'bytes',
        'sha256',
        'reason',
    ],
).to_csv(
    DURABLE_INVENTORY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Human-readable README
# ------------------------------------------------------------

README_PATH = (
    UPLOAD
    / 'README.txt'
)

readme_lines = [
    '59R2 COMPONENT-INVARIANT REMOVE CHALLENGER',
    'FINAL REVIEW PACKAGE',
    '',

    'STATUS',
    '------',

    'Verdict: CHALLENGER_BUILT',

    'Promotion authority: False',

    (
        'This package does not promote 59R2 over '
        'the frozen 59R1 incumbent.'
    ),

    '',

    'IDENTITY',
    '--------',

    f'Policy ID: '
    f'{policy["policy_id"]}',

    f'Compatibility ID: '
    f'{policy["compatibility_id"]}',

    f'Bundle version: '
    f'{freeze_manifest["bundle_version"]}',

    f'Freeze manifest SHA-256: '
    f'{sha256_file(FREEZE_MANIFEST_PATH)}',

    '',

    'CORE RESULTS',
    '------------',

    f'Complete reviewed cohort pages: '
    f'{challenger_verdict["complete_reviewed_cohort_pages"]}',

    f'Complete reviewed cohort pixels: '
    f'{challenger_verdict["complete_reviewed_cohort_pixels"\]:,}',

    f'59R1 coherent SAFE_REMOVE pixels: '
    f'{challenger_verdict["59r1_safe_remove_pixels"\]:,}',

    f'59R2 coherent SAFE_REMOVE pixels: '
    f'{challenger_verdict["59r2_safe_remove_pixels"\]:,}',

    f'59R2 - 59R1 SAFE_REMOVE delta: '
    f'{challenger_verdict["safe_remove_delta"\]:,}',

    f'Changed pixel decisions: '
    f'{challenger_verdict["changed_pixel_decisions"\]:,}',

    '',

    'INTEGRITY',
    '---------',

    f'Ordinary exact replay passed: '
    f'{replay_verdict["passed"]}',

    f'Component-renumbering invariance passed: '
    f'{invariance_verdict["passed"]}',

    f'Renumbering changed full-cohort decisions: '
    f'{invariance_verdict["changed_full_cohort_decisions_total"]}',

    f'Renumbering changed coherent-output pixels: '
    f'{invariance_verdict["changed_coherent_output_pixels_total"]}',

    'component_id predictor: False',

    'component_key predictor: False',

    (
        'Component identity role: '
        'audit/join key only'
    ),

    '',

    'PACKAGE CONTENT',
    '---------------',

    (
        'bundle/: frozen policy, predictor schema, '
        'runtime contract, freeze manifest and freeze verdict'
    ),

    (
        'replay/: ordinary exact replay verdict'
    ),

    (
        'audit/: source hashes, computational/memory record, '
        'component-renumbering mappings, replay sample, '
        'invariance table and verdict'
    ),

    (
        'tables/: compact model-development evidence'
    ),

    (
        'comparison_59R1/: complete reviewed-cohort '
        'incumbent/challenger comparison'
    ),

    (
        'charts/: concise comparison graphics'
    ),

    '',

    'HEAVYWEIGHT OBJECTS',
    '-------------------',

    (
        'The model, calibrator, large pixel tables and '
        'working arrays are not duplicated into this review ZIP.'
    ),

    (
        'They remain in the durable 59R2 run and are '
        'content-addressed by freeze_manifest.json and/or '
        'durable_not_copied.csv.'
    ),

    '',

    'INTERPRETATION',
    '--------------',

    (
        '59R2 is a separately versioned component-invariant '
        'challenger, not a runtime patch to 59R1.'
    ),

    (
        'Automatic promotion is prohibited. The next gate is '
        'independent safety/benefit review, source-group '
        'stability assessment and a documented promotion '
        'decision.'
    ),
]

README_PATH.write_text(
    chr(10).join(
        readme_lines
    )
    + chr(10)
)


# ------------------------------------------------------------
# 13. Machine-readable package identity
# ------------------------------------------------------------

PACKAGE_IDENTITY_PATH = (
    UPLOAD
    / 'upload_package_identity.json'
)

package_identity = {
    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'run':
        str(
            RUN
        ),

    'package_type':
        '59R2 challenger review package',

    'challenger_verdict':
        challenger_verdict[
            'verdict'
        ],

    'promotion_authority':
        False,

    'policy_id':
        policy[
            'policy_id'
        ],

    'compatibility_id':
        policy[
            'compatibility_id'
        ],

    'bundle_version':
        freeze_manifest[
            'bundle_version'
        ],

    'freeze_manifest_sha256':
        sha256_file(
            FREEZE_MANIFEST_PATH
        ),

    'freeze_verdict_sha256':
        sha256_file(
            FREEZE_VERDICT_PATH
        ),

    'runtime_contract_sha256':
        sha256_file(
            RUNTIME_CONTRACT_PATH
        ),

    'ordinary_replay_passed':
        bool(
            replay_verdict[
                'passed'
            ]
        ),

    'renumbering_invariance_passed':
        bool(
            invariance_verdict[
                'passed'
            ]
        ),

    'component_identity_is_predictor':
        False,

    'required_review_artefacts':
        len(
            REQUIRED_UPLOAD_ARTEFACTS
        ),

    'optional_review_artefacts_packaged':
        len(
            optional_packaged
        ),

    'optional_review_artefacts_missing':
        optional_missing,

    'durable_heavyweight_objects_not_copied':
        len(
            durable_not_copied
        ),

    'source_artefacts_unchanged':
        True,
}

atomic_json(
    PACKAGE_IDENTITY_PATH,
    package_identity,
)


# ------------------------------------------------------------
# 14. Upload file list
#
# Generate this only after README, package identity and durable
# inventory exist.
# ------------------------------------------------------------

UPLOAD_FILE_LIST_PATH = (
    UPLOAD
    / 'upload_file_list.txt'
)

package_files_before_zip = sorted(
    str(
        path.relative_to(
            UPLOAD
        )
    )
    for path in UPLOAD.rglob('*')
    if path.is_file()
)

UPLOAD_FILE_LIST_PATH.write_text(
    chr(10).join(
        package_files_before_zip
    )
    + chr(10)
)


# ------------------------------------------------------------
# 15. Pre-ZIP validation
# ------------------------------------------------------------

for row in upload_manifest.itertuples(
    index=False
):

    packaged = (
        UPLOAD
        / row.relative_path
    )
equire(
        packaged.is_file(),
        (
            'Packaged file missing before ZIP creation: '
            f'{row.relative_path}'
        ),
    )

    require(
        sha256_file(
            packaged
        )
        == row.sha256,
        (
            'Packaged file hash mismatch before ZIP: '
            f'{row.relative_path}'
        ),
    )

    require(
        int(
            packaged.stat().st_size
        )
        == int(
            row.bytes
        ),
        (
            'Packaged file size mismatch before ZIP: '
            f'{row.relative_path}'
        ),
    )


# ------------------------------------------------------------
# 16. Create final ZIP outside upload/
#
# Keeping the ZIP outside UPLOAD prevents accidental
# self-inclusion.
# ------------------------------------------------------------

zip_path = (
    RUN
    / '59R2_Component_Invariant_REMOVE_Challenger_upload_bundle.zip'
)

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for path in sorted(
        UPLOAD.rglob('*')
    ):

        if path.is_file():

            zf.write(
                path,
                path.relative_to(
                    UPLOAD
                ),
            )


require(
    zip_path.is_file(),
    'Final Upload Package ZIP was not created.',
)

require(
    zip_path.stat().st_size > 0,
    'Final Upload Package ZIP is empty.',
)


# ------------------------------------------------------------
# 17. Verify ZIP contents against staged package
# ------------------------------------------------------------

with zipfile.ZipFile(
    zip_path,
    'r',
) as zf:

    zipped_files = sorted(
        name
        for name in zf.namelist()
        if not name.endswith('/')
    )

    staged_files = sorted(
        str(
            path.relative_to(
                UPLOAD
            )
        )
        for path in UPLOAD.rglob('*')
        if path.is_file()
    )

    require(
        zipped_files
        == staged_files,
        (
            'ZIP file list does not exactly match '
            'the final staged upload directory.'
        ),
    )

    bad_zip_members = (
        zf.testzip()
    )

    require(
        bad_zip_members is None,
        (
            'ZIP integrity failure at member: '
            f'{bad_zip_members}'
        ),
    )


# ------------------------------------------------------------
# 18. Write final package verdict beside the ZIP
# ------------------------------------------------------------

PACKAGE_VERDICT_PATH = (
    RUN
    / 'upload_package_verdict.json'
)

package_verdict = {
    'passed':
        True,

    'package_type':
        '59R2 component-invariant challenger review',

    'zip_path':
        str(
            zip_path
        ),

    'zip_bytes':
        int(
            zip_path.stat().st_size
        ),

    'zip_sha256':
        sha256_file(
            zip_path
        ),

    'packaged_files':
        len(
            staged_files
        ),

    'required_artifacts_all_present':
        True,

    'packaged_hashes_verified':
        True,

    'zip_file_list_exact':
        True,

    'zip_integrity_passed':
        True,

    'freeze_verdict_passed':
        True,

    'ordinary_replay_passed':
        True,

    'component_renumbering_invariance_passed':
        True,

    'component_identity_is_predictor':
        False,

    'promotion_authority':
        False,
}

atomic_json(
    PACKAGE_VERDICT_PATH,
    package_verdict,
)


# ------------------------------------------------------------
# 19. Console summary
# ------------------------------------------------------------

print(
    '\n59R2 FINAL UPLOAD PACKAGE: PASS'
)

print(
    'Packaged review files:',
    len(
        staged_files
    ),
)

print(
    'Required artefacts:',
    len(
        REQUIRED_UPLOAD_ARTEFACTS
    ),
)

print(
    'Optional artefacts included:',
    len(
        optional_packaged
    ),
)

print(
    'Optional artefacts absent:',
    len(
        optional_missing
    ),
)

print(
    'Durable heavyweight objects not copied:',
    len(
        durable_not_copied
    ),
)

print(
    'ZIP:',
    zip_path,
)

print(
    'ZIP size:',
    f'{zip_path.stat().st_size / 1024**2:.2f} MiB',
)

print(
    'ZIP SHA-256:',
    sha256_file(
        zip_path
    ),
)

print(
    'Promotion authority:',
    False,
)

print(
    '\nPackage verdict:'
)

print(
    json.dumps(
        package_verdict,
        indent=2,
    )
)